Preprocess data for recommender setup. 
* Also subsets data according to clinical trial task setup. (and subsets cointexts by that also).
* Merges some sparse columns together into single text col


tf-rs +keras example, 2 tower: expedia:
* https://medium.com/expedia-group-tech/candidate-generation-using-a-two-tower-approach-with-expedia-group-traveler-data-ca6a0dcab83e

keras-rs, tensorflow recommenders ?
* deepmatch, deepctr, ms recc;
* https://github.com/shenweichen/DeepMatch/blob/master/examples/colab_MovieLen1M_SDM.ipynb
* Maybe try LibRecommender (seems useful): https://librecommender.readthedocs.io/en/latest/user_guide/feature_engineering.html
    * libReco doesn't have support for text features!

Pretrained GO embeddings - could use anc2vec or others

STRING (+- protein) embeddings: https://github.com/deweihu96/SPACE
* https://chatgpt.com/share/68a30d02-0fd4-8013-aadc-5f2e475a9c69 
* human‑specific string embedding: `9606.protein.network.embeddings.v12.0.h5`. Contains two datasets – embeddings (N×D array) and proteins (a list of STRING protein identifiers)
    * STRING  alias file: `9606.protein.aliases.v12.0.txt.gz` maps each STRING protein ID to various aliases (including Ensembl gene IDs).
* https://github.com/krishnanlab/PecanPy - fast training
* network features, embeddings: biosnap? STRING, bioGrid
    * https://snap.stanford.edu/biodata/index.html
 
Pretrained text embeddings - (I don't understand the code + it's on different (news) data). 
* https://github.com/ebanalyse/ebnerd-benchmark/blob/main/src/ebrec/models/newsrec/nrms_docvec.py 
* https://github.com/ebanalyse/ebnerd-benchmark/blob/main/examples/quick_start/nrms_ebnerd.ipynb
* https://github.com/ebanalyse/ebnerd-benchmark/blob/17d2f308073389cb7f4bd40c3136e891706ff721/src/ebrec/utils/_articles.py#L21
* https://github.com/recommenders-team/recommenders/blob/main/examples/02_model_content_based_filtering/dkn_deep_dive.ipynb

In [109]:
import os
import numpy as np
import pandas as pd
import shap
# from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split, GroupKFold, StratifiedGroupKFold
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, f1_score, accuracy_score, precision_score, recall_score

# import tensorflow as tf, keras

# DATA_DIR = "./opentargets/"
DATA_DIR = "../data/opentargets/"

In [110]:
SAVE_INTERMEDIATES =False
SAVE_NOVEL_PREDICTION_CANDIDATES = False 

FAST_RUN = False#True
# FAST_RUN = True

RUN_CV = True

In [111]:
import json, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, make_scorer, average_precision_score
from catboost import CatBoostClassifier, Pool, cv

# ------------------------ config ---------------------------------
CFG = dict(
    parquet_path   = "df_sample.parquet",
    target_col     = "label",
    embed_cols     = ["diseaseEmbeddings", "targetEmbeddings"],
    text_cols      = ["description",
                      "ancestors",
                      # "therapeuticAreas",
                      "go",
                      # "synonyms",
                      "functionDescriptions","subcellularLocations",
                      # "pathways", "targetClass",
                      "tractability",
                      # 'dbXRefs',
                     ],
    cat_cols       = ["diseaseId","targetId","biotype"],
    cb_params      = dict(iterations=30 if FAST_RUN else 1000, depth=4 if FAST_RUN else 8,#depth=10 , ~ 2500 iter # for good perf
                          loss_function="Logloss",
                          eval_metric="AUC", # not supported on gpu? 
                          early_stopping_rounds=50, #verbose=False,
                          task_type="GPU", ## runs out of mem with many cols
                          random_seed=42,
                         auto_class_weights = "SqrtBalanced",
                         # dictionaries= ['Word:min_token_occurrence=5']#,'BiGram:gram_order=2'
                          # max_ctr_complexity = 3,
                        ),
    cv_folds       = 5,
    # max_ctr_complexity = 3, # default 4 ? 
)


cat_features       = ["diseaseId","biotype"] # ,"biotype"
# text_features      = ["disease_text","target_text"] # # could maybe use sparse tfidf feats instead +- embedding features
# embed_cols     = ["diseaseEmbeddings", "targetEmbeddings"] # need to keep ID col + merge. ## Note: embeddings = heavy compute! 
embed_cols = None

text_features = ["disease_text","target_text",
                 # 'dbXRefs','therapeuticAreas', ## now merged in text
                 "go","subcellularLocations",
                 "modelPhenotypeClasses","modelPhenotypeId", ## mouse feat
                 "phenotypes","tractability",
                 'ancestors', 
                ]
# text_features      = ["disease_text",] 
# embed_cols     = [ "targetEmbeddings"] # need to keep ID col + merge. ## Note: embeddings = heavy compute! 

In [112]:
def preprocess_open_targets(df: pd.DataFrame, top_n_tissues: int = 5) -> pd.DataFrame:
    """Preprocess nested DepMap essentiality data into a flat feature table.
    >>> target_essentiality = pd.read_parquet(os.path.join(Config.DATA_DIR, 'target_essentiality'))
    >>> preprocess_open_targets(target_essentiality.head())
    """
    # Determine globally most frequent tissues from context-level data
    all_tissues = []
    for gene_ess_list in df["geneEssentiality"]:
        for gene_ess in gene_ess_list or []:
            for dep in gene_ess.get("depMapEssentiality", []):
                if dep.get("tissueName"):
                    all_tissues.append(dep["tissueName"])
    top_tissues = (
        pd.Series(all_tissues).value_counts().index.tolist()[:top_n_tissues]
    )

    result_records = []
    for _, row in df.iterrows():
        gene_id = row["id"]
        contexts = []
        screens = []

        # flatten contexts and screens; propagate tissueName to each screen
        for gene_ess in row.get("geneEssentiality") or []:
            for dep in gene_ess.get("depMapEssentiality", []):
                contexts.append(dep)
                tissue = dep.get("tissueName")
                for scr in dep.get("screens", []):
                    scr_aug = scr.copy()
                    scr_aug["tissueName"] = tissue  # attach context tissue to screen
                    screens.append(scr_aug)

        if not contexts:
            continue

        # context-level essentiality metrics
        n_contexts = len(contexts)
        n_essential_contexts = sum(1 for c in contexts if c.get("isEssential"))
        essential_ratio = n_essential_contexts / n_contexts

        # screen-level statistics
        screens_df = pd.DataFrame(screens)
        avg_gene_effect = screens_df["geneEffect"].mean()
        std_gene_effect = screens_df["geneEffect"].std(ddof=0)
        min_gene_effect = screens_df["geneEffect"].min()
        max_gene_effect = screens_df["geneEffect"].max()
        avg_expression = screens_df["expression"].mean()
        max_expression = screens_df["expression"].max()
        n_mutations = screens_df["mutation"].nunique()
        n_diseases = (
            screens_df["diseaseFromSource"].nunique()
            if "diseaseFromSource" in screens_df
            else None
        )

        context_tissues = [c.get("tissueName") for c in contexts if c.get("tissueName")]
        n_tissues = len(set(context_tissues))

        result = {
            "id": gene_id,
            # "n_contexts": n_contexts,
            "n_essential_contexts": n_essential_contexts,
            "essential_ratio": essential_ratio,
            "avg_gene_effect": avg_gene_effect,
            "std_gene_effect": std_gene_effect,
            "min_gene_effect": min_gene_effect,
            "max_gene_effect": max_gene_effect,
            "avg_expression": avg_expression,
            "max_expression": max_expression,
            "n_tissues": n_tissues,
            "n_mutations": n_mutations,
            "n_diseases": n_diseases,
        }

        # tissue-specific aggregations (context- and screen-level)
        for tissue in top_tissues:
            # context counts and essential ratios
            n_contexts_tissue = sum(
                1 for c in contexts if c.get("tissueName") == tissue
            )
            n_essential_tissue = sum(
                1
                for c in contexts
                if c.get("tissueName") == tissue and c.get("isEssential")
            )
            # result[f"context_count_{tissue}"] = n_contexts_tissue
            result[f"context_essential_ratio_{tissue}"] = (
                n_essential_tissue / n_contexts_tissue if n_contexts_tissue else None
            )

            # screen-level counts and averages by tissue
            tissue_screens = screens_df[screens_df["tissueName"] == tissue]
            # result[f"screen_count_{tissue}"] = len(tissue_screens)
            result[f"avg_gene_effect_{tissue}"] = (
                tissue_screens["geneEffect"].mean()
                if not tissue_screens.empty
                else None
            )

        result_records.append(result)
    result_records = pd.DataFrame(result_records)
    result_records = result_records.loc[:,result_records.nunique(dropna=False)!=1]
    return result_records


from sklearn.model_selection import StratifiedGroupKFold
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, f1_score, precision_score, recall_score
)

def run_catboost_cv(
    df,
    cat_features,
    text_features,
    embed_cols,
    cb_params,
    n_splits=5,
    group_col="targetId",
    merge_aba=False,
    random_state=42,
    return_model=True,  # model returned is a little different than not using pool..
    MINIMAL_FEATURES=True,
):
    """
    Single-run groupwise CV with CatBoost, splitting by group_col.

    Returns:
        oof_df  : DataFrame with columns ["pred", "label", group_col], aligned to df.index
        metrics_df : per-fold metrics
        summary    : dict with mean metrics across folds
        model      : last fitted CatBoost model (if return_model=True)
    """
    df = df.copy()
    s0 = df.shape[0]
    y_all = df["label"].astype(int)

    # Build features (may reorder/drop rows)
    X = get_cb_x(df, merge_aba=merge_aba,
                get_embeddings=False,
                add_target_feats=False,
                merge_mouse_pheno=False,
                merge_depmap_essentiality=False,
                get_network_embeds=False,
                embed_cols=embed_cols)
    assert s0 == X.shape[0], f"s0{s0} != X.shape[0]{X.shape[0]}"
    if MINIMAL_FEATURES:
        X = X.filter(['diseaseId', 'disease_text', 'target_text'],axis=1)
    # Keep only rows present in BOTH (preserve X's order for split indices)
    idx = X.index.intersection(df.index)
    X = X.loc[idx]
    assert s0 == X.shape[0], f"s0{s0} != X.shape[0]{X.shape[0]}; after index intersection"

    y = y_all.loc[idx]
    groups = df.loc[idx, group_col].values

    print(X.shape, "- X.shape")
    print(X.columns)

    gkf = StratifiedGroupKFold(
        n_splits=n_splits,
        random_state=random_state,
        shuffle=True,
    )

    # OOF over the *X* rows (positional)
    oof_pos = np.full(len(X), np.nan, dtype=float)
    fold_metrics = []
    last_model = None

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val,   y_val   = X.iloc[val_idx],   y.iloc[val_idx]

        train_pool = Pool(
            X_train, y_train,
            cat_features=cat_features,
            text_features=text_features,
            embedding_features=embed_cols,
        )
        val_pool = Pool(
            X_val, y_val,
            cat_features=cat_features,
            text_features=text_features,
            embedding_features=embed_cols,
        )

        model = CatBoostClassifier(**cb_params)
        model.fit(train_pool, verbose=200)  # optionally add eval_set / early stopping
        last_model = model

        preds = model.predict_proba(val_pool)[:, 1]
        oof_pos[val_idx] = preds  # positional write

        metrics = {
            "fold": fold,
            "roc_auc": roc_auc_score(y_val, preds),
            "prauc": average_precision_score(y_val, preds),
            "accuracy": accuracy_score(y_val, preds > 0.5),
            "f1": f1_score(y_val, preds > 0.5),
            "precision": precision_score(y_val, preds > 0.5),
            "recall": recall_score(y_val, preds > 0.5),
        }
        fold_metrics.append(metrics)

    metrics_df = pd.DataFrame(fold_metrics)
    summary = metrics_df.mean(numeric_only=True).round(4).to_dict()
    print(summary)

    # Expand OOF back to full df index (NaN where rows weren’t in X)
    # Compact OOF (only rows used by X)
    oof_series_compact = pd.Series(oof_pos, index=idx, name="pred")

    # Expand back to full df index, NaN where row wasn't in X (shouldn't happen, but safe)
    oof_series_full = pd.Series(np.nan, index=df.index, name="pred")
    oof_series_full.loc[idx] = oof_series_compact.values

    # ✅ Return the full original dataframe + OOF predictions
    # This preserves diseaseId, disease_text, source, etc.
    oof_df = df.copy()
    oof_df["pred"] = oof_series_full

    if return_model:
        return oof_df, metrics_df, summary, model
    else:
        return oof_df, metrics_df, summary


In [113]:
def reset_pivoted_multiindex(df):
    df = df.sort_index(axis=1, level=1)
    df.columns = [f'{x}_{y}' for x,y in df.columns]
    df = df.reset_index()
    return df

##modified (index stuff):
def get_cb_x(df,
             keep_col=["diseaseId","disease_text","target_text"],
             drop_later_cols=["targetId"],
             get_embeddings=False,
             add_target_feats=True,
             merge_aba=False,
             merge_mouse_pheno=False,
             merge_depmap_essentiality=True,
             # get_network_embeds = None#True
             get_network_embeds = True
             ,embed_cols=None):  # <-- add this param explicitly (it was referenced)
    s_base = df.shape[0]
    df = df.copy()
    # carry a stable row id through merges
    df["_rid"] = df.index

    df = df.drop(columns=["label"], errors="ignore").filter(keep_col + drop_later_cols + ["_rid"], axis=1)

    if get_embeddings: 
        df = df.merge(df_embed_target, on="targetId", how="left")
        df = df.merge(df_embed_disease, on="diseaseId", how="left")

    if add_target_feats:
        if "id" in target_df.columns:
            target_df.rename(columns={"id":"targetId"}, inplace=True)
        df = df.merge(
            target_df.set_index("targetId")
                     .filter(target_df.select_dtypes(["number","bool"]).columns.tolist()
                             + ["subcellularLocations","biotype",], axis=1), # "tractability","go", already in text col  
            on="targetId", how="left"
        )
        df = df.merge(target_priority, on="targetId", how="left") # merge with target_priority features

    # disease-level extras
    df = df.merge(df_disease_pheno, on="diseaseId", how="left")
    df["diseaseHasPheno"] = df["diseaseId"].isin(df_disease_pheno["diseaseId"])

    if merge_mouse_pheno:
        df = df.merge(df_mouse_pheno_agg, on="targetId", how="left")
        df["TargetHasMousePheno"] = df["targetId"].isin(df_mouse_pheno_agg["targetId"])

    df = df.merge(
        disease_df[["diseaseId","ancestors",
                    # "therapeuticAreas","dbXRefs", ## already merged into in text col
                   ]],
        on="diseaseId", how="left"
    ).drop(columns=["id"], errors="ignore")

    if merge_depmap_essentiality:
        df = df.merge(target_essentiality, on="targetId", how="left")

    if merge_aba:
        s1 = df.shape[1]; s0 = df.shape[0]
        df = df.merge(df_aba.select_dtypes("number"), left_on="targetId", right_index=True, how="left")
        df["in_aba"] = df["targetId"].isin(df_aba.index).astype(int)
        print("in_aba\n", df["in_aba"].agg(["mean","sum","count"]).round(3))
        print(df.shape[1] - s1, "ABA columns added")
        assert s1 < df.shape[1]
        assert s0 == df.shape[0], f"rows mismatch from aba; premerge-aba: {s0} vs new: {df.shape[0]}"

    if get_network_embeds:
        df = df.merge(df_targ_network_embed, on="targetId", how="left")

    # clean strings (respect embed_cols if provided)
    for c in df.select_dtypes(["O","string"]).columns:
        if c == "_rid":
            continue
        if embed_cols is not None:
            if c not in embed_cols:
                df[c] = df[c].fillna("").astype(str)
        else:
            df[c] = df[c].fillna("").astype(str)

    # restore original index and drop helper cols
    assert s_base == df.shape[0], f"rows mismatch; expected {s_base}, got {df.shape[0]}"
    df = df.set_index("_rid").sort_index()
    df.drop(columns=drop_later_cols, errors="ignore", inplace=True)

    print(df.shape)
    return df

In [114]:
# # 01_prepare_data.py
# # GOAL: To perform all heavy, one-time data processing and feature engineering.
# # This script is independent of the modeling library and produces clean, ML-ready files.
# ## run in advance 

class Config:
    DATA_DIR = "../data/opentargets/"
    PROCESSED_DATA_FILE = 'final_df.parquet'
    DISEASE_EMBEDDINGS_FILE = 'disease_embeddings.npz'
    TARGET_EMBEDDINGS_FILE = 'target_embeddings.npz'
    TEXT_MODEL = "nomic-ai/modernbert-embed-base"#'sentence-transformers/all-MiniLM-L6-v2'
# model = SentenceTransformer("nomic-ai/modernbert-embed-base")

In [115]:
## note: gets updated later according to our df_int/learn targets data!
druggable_genome_list = pd.read_csv(os.path.join("../data", "finan_proc_druggable_genome_list.csv"))["ensembl_gene_id"].to_list()
print(len(druggable_genome_list))

## unique_targets 
unique_targets = druggable_genome_list
print(len(unique_targets),"# druggable genome targets")

4479
4479 # druggable genome targets


* Target pipeline

In [116]:
disease_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'disease')).filter(['id','name', 'description', 
                                                                               'dbXRefs',
                                                                               'synonyms',
                                                                               'ancestors',"parents",'therapeuticAreas'],axis=1).dropna(axis=1,how="all")
print(disease_df["id"].nunique(),"# unique Diseases")
# #ORIG
# # ## max 5 each. 
# disease_df["ExactSynonyms"] = disease_df["synonyms"].str.lower().apply(lambda d: list(dict.fromkeys(d.get("hasExactSynonym", [])))
#                         if isinstance(d, dict) else []) #PREV
disease_df["ExactSynonyms"] = disease_df["synonyms"].apply(lambda d: list(dict.fromkeys(d.get("hasExactSynonym", [])))[0:8]
                        )
disease_df["ExactSynonyms"] = disease_df["ExactSynonyms"].apply(" ".join).str.strip() # make str. 
# disease_df.drop(columns=["synonyms"],inplace=True,errors="ignore")

for c in disease_df.select_dtypes("O").columns:
    print(c)
    try:
        print(disease_df[c].str.len().describe().round(1),"len")
        ## truncate. Note: this is different for strings (description) vs lists! (# items)
        # Replaces: disease_df[c] = disease_df[c].str.slice(0,2_000)  # ORIG char based
        disease_df[c] = disease_df[c].apply(lambda x: " ".join(x.split()[:2_500]) if isinstance(x, str) else x[:1_000] if isinstance(x, list) else x)
    except:
        print("ERROR",c)

#Add joint text description col, used for embeddings.
## id is only relevant if using pretrained embeddings 
# 'id', 
# ##ORIG:
# disease_df["disease_text_embed"] = disease_df[['name',"ExactSynonyms","description"]].astype(str).apply(" ".join,axis=1)
##expanded text: 
disease_df["disease_text_embed"] = disease_df[['name',"ExactSynonyms","description",'dbXRefs','therapeuticAreas',"parents"]].astype(str).apply(" ".join,axis=1)
# disease_df.rename(columns={"id":"diseaseId"},inplace=True) # do it later ,for compat with target pipeline code
disease_df

38959 # unique Diseases
id
count    38959.0
mean        11.7
std          1.1
min          9.0
25%         11.0
50%         11.0
75%         13.0
max         15.0
Name: id, dtype: float64 len
name
count    38959.0
mean        38.5
std         17.9
min          3.0
25%         25.0
50%         37.0
75%         49.0
max        153.0
Name: name, dtype: float64 len
description
count    34311.0
mean       151.6
std        144.3
min          1.0
25%         67.0
50%         92.0
75%        180.0
max       2314.0
Name: description, dtype: float64 len
dbXRefs
count    38959.0
mean         3.1
std          3.8
min          0.0
25%          1.0
50%          1.0
75%          5.0
max         60.0
Name: dbXRefs, dtype: float64 len
synonyms
count    38959.0
mean         4.0
std          0.0
min          4.0
25%          4.0
50%          4.0
75%          4.0
max          4.0
Name: synonyms, dtype: float64 len
ancestors
count    38959.0
mean         6.3
std          6.7
min          0.0
25%          2

,id,name,description,dbXRefs,synonyms,ancestors,parents,therapeuticAreas,ExactSynonyms,disease_text_embed
0,DOID_0050890,synucleinopathy,A neurodegenerative disease that is characteri...,"[MESH:D000080874, MONDO:0000510, UMLS:C5191670...",{'hasExactSynonym': ['alpha Synucleinopathies'...,"[MONDO_0024237, EFO_0005772, EFO_0000618, MOND...","[MONDO_0019052, MONDO_0021179, MONDO_0024237]","[EFO_0000618, OTAR_0000018, OTAR_0000020]",alpha Synucleinopathies synucleinopathy,synucleinopathy alpha Synucleinopathies synucl...
1,DOID_10113,trypanosomiasis,Infection with protozoa of the genus trypanosoma.,"[UMLS:C0041227, MONDO:0000940, ICD10CM:B56, Me...",{'hasExactSynonym': ['Trypanosoma caused disea...,"[MONDO_0002428, EFO_0001067, EFO_0005741]",[MONDO_0002428],[EFO_0005741],Trypanosoma caused disease or disorder Trypano...,trypanosomiasis Trypanosoma caused disease or ...
2,DOID_10718,giardiasis,An infection of the small intestine caused by ...,"[MeSH:D005873, ICD9CM:007.1, MESH:D005873, MON...","{'hasExactSynonym': ['giardiasis', 'beaver fea...","[EFO_0010282, EFO_0009431, EFO_0009561, EFO_00...","[MONDO_0002428, EFO_0009561]","[EFO_0010282, EFO_0005741]",giardiasis beaver feaver beaver fever infectio...,giardiasis giardiasis beaver feaver beaver fev...
3,DOID_13406,pulmonary sarcoidosis,Sarcoidosis affecting the lung parenchyma. It ...,"[SNOMEDCT:187230004, UMLS:C0036205, MONDO:0001...","{'hasExactSynonym': ['Sarcoidosis, Pulmonary',...","[MONDO_0017026, MONDO_0019338, EFO_0004244, EF...","[MONDO_0017026, MONDO_0019338]","[OTAR_0000010, OTAR_0000006]","Sarcoidosis, Pulmonary sarcoidosis of lung lun...","pulmonary sarcoidosis Sarcoidosis, Pulmonary s..."
4,DOID_1947,trichomoniasis,An infection that is caused by Trichomonas.,"[ICD9:131.8, MESH:D014245, ICD10CM:A59, ICD9CM...","{'hasExactSynonym': ['trichomoniasis', 'Tricho...","[EFO_0005741, EFO_0001067, MONDO_0002428]",[MONDO_0002428],[EFO_0005741],trichomoniasis Trichomonas Infections Trichomo...,trichomoniasis trichomoniasis Trichomonas Infe...
...,...,...,...,...,...,...,...,...,...,...
38954,Orphanet_99942,Autosomal dominant Charcot-Marie-Tooth disease...,Autosomal dominant Charcot-Marie-Tooth disease...,"[ICD10:G60.0, OMIM:607677]","{'hasExactSynonym': ['CMT2I'], 'hasRelatedSyno...","[OTAR_0000018, EFO_0003100, EFO_0000618, MONDO...",[MONDO_0018993],"[OTAR_0000018, EFO_0000618]",CMT2I,Autosomal dominant Charcot-Marie-Tooth disease...
38955,Orphanet_99943,Autosomal dominant Charcot-Marie-Tooth disease...,Autosomal dominant Charcot-Marie-Tooth disease...,"[ICD10:G60.0, OMIM:607736]","{'hasExactSynonym': ['CMT2J'], 'hasRelatedSyno...","[EFO_1001902, EFO_0009387, EFO_0004149, MONDO_...",[MONDO_0018993],"[EFO_0000618, OTAR_0000018]",CMT2J,Autosomal dominant Charcot-Marie-Tooth disease...
38956,Orphanet_99945,Autosomal dominant Charcot-Marie-Tooth disease...,Autosomal dominant Charcot-Marie-Tooth disease...,"[ICD10:G60.0, OMIM:608673]","{'hasExactSynonym': ['CMT2L'], 'hasRelatedSyno...","[EFO_1001902, EFO_0003100, EFO_0000508, MONDO_...",[MONDO_0018993],"[OTAR_0000018, EFO_0000618]",CMT2L,Autosomal dominant Charcot-Marie-Tooth disease...
38957,Orphanet_99946,Autosomal dominant Charcot-Marie-Tooth disease...,Autosomal dominant Charcot-Marie-Tooth disease...,"[ICD10:G60.0, OMIM:118210]","{'hasExactSynonym': ['CMT2A1'], 'hasRelatedSyn...","[MONDO_0100546, MONDO_0020127, EFO_0003100, EF...",[MONDO_0018993],"[EFO_0000618, OTAR_0000018]",CMT2A1,Autosomal dominant Charcot-Marie-Tooth disease...


In [117]:
# disease_df.loc[disease_df["name"].str.contains("alzheimer",case=False)]
disease_df["disease_text_embed"].str.split().str.len().describe().round(1) # max 350

count    38959.0
mean        39.1
std         26.3
min          5.0
25%         20.0
50%         32.0
75%         53.0
max        358.0
Name: disease_text_embed, dtype: float64

In [118]:
disease_df.loc[disease_df["name"].str.contains("alzheimer",case=False)]["disease_text_embed"].iloc[0]

"Alzheimer disease presenile and senile dementia Alzheimer's dementia Alzheimers disease AD Alzheimer dementia Alzheimer disease Alzheimers dementia Alzheimer's disease A progressive, neurodegenerative disease characterized by loss of function and death of nerve cells in several areas of the brain leading to loss of cognitive function such as memory and language. ['SCTID:142811000119104' 'ICD9:331.0' 'NCIT:C2866' 'NIFSTD:birnlex_2092'\n 'MESH:D000544' 'ICD9:290.1' 'MEDGEN:1853' 'DOID:10652' 'HP:0002511'\n 'UMLS:C0002395' 'Orphanet:238616' 'icd11.foundation:1611724421'\n 'ICD10CM:G30'] ['MONDO_0002025' 'EFO_0000618'] ['MONDO_0001627' 'EFO_0005815']"

In [119]:
print(disease_df["ExactSynonyms"].drop_duplicates().iloc[1])

Trypanosoma caused disease or disorder Trypanosoma infectious disease Trypanosoma disease or disorder trypanosomiasis


### More disease - phenotype values
* count encoded
 * replace rare vals
  * non humanized values

In [120]:
df_disease_pheno = pd.read_parquet(DATA_DIR+"disease_phenotype",columns=["disease","phenotype"]).rename(columns={"disease":"diseaseId"}).drop_duplicates()
# print(df_disease_pheno.shape[0])
# df_disease_pheno = df_disease_pheno.loc[df_disease_pheno["diseaseId"].isin(unique_diseases)]
print(df_disease_pheno.shape[0],"# rows after filtering by (valid) diseaseIds)")
print(df_disease_pheno.nunique())

#### count encoding
counts = df_disease_pheno['phenotype'].value_counts()
threshold = 3  # Example: phenotype/categories appearing 3 or less times are rare
rare_categories = counts[counts <= threshold].index
df_disease_pheno['phenotype'] = df_disease_pheno['phenotype'].replace(rare_categories, 'Rare').fillna("Unknown_Phenotypes")

# df_disease_pheno.groupby("phenotype").count().describe().round()

df_disease_pheno = df_disease_pheno.groupby("diseaseId").agg(phenotypes_count=('phenotype', 'count'),  # Get the count of 'Value' for each group
        phenotypes=('phenotype', lambda x: ' '.join(x)) 
    ).reset_index()
df_disease_pheno

157611 # rows after filtering by (valid) diseaseIds)
diseaseId    6665
phenotype    8670
dtype: int64


,diseaseId,phenotypes_count,phenotypes
0,EFO_0000095,3,HP_0001442 Rare Rare
1,EFO_0000174,2,HP_0001442 Rare
2,EFO_0000178,3,HP_0001442 HP_0012126 HP_0410067
3,EFO_0000181,2,HP_0000007 HP_0002860
4,EFO_0000182,4,HP_0001442 Rare HP_0001402 HP_0001413
...,...,...,...
6660,Orphanet_99942,11,HP_0002936 HP_0002460 HP_0003484 HP_0001284 HP...
6661,Orphanet_99943,17,HP_0003376 HP_0000408 HP_0001265 HP_0002460 HP...
6662,Orphanet_99945,14,HP_0002936 HP_0011462 HP_0002650 HP_0001265 HP...
6663,Orphanet_99946,21,HP_0001765 HP_0003431 HP_0003384 HP_0010628 HP...


In [121]:
disease_df = disease_df.merge(df_disease_pheno[["diseaseId","phenotypes"]], left_on="id",right_on="diseaseId", how="left")
disease_df["phenotypes"] = disease_df["phenotypes"].fillna("Unknown_Phenotypes")
disease_df["disease_text_embed"] = (disease_df["disease_text_embed"]+" "+ disease_df["phenotypes"])
disease_df.drop(columns=["diseaseId","phenotypes"],inplace=True,errors="ignore") # we add diseaseId in later


In [122]:
disease_df["disease_text_embed"].iloc[0]

"synucleinopathy alpha Synucleinopathies synucleinopathy A neurodegenerative disease that is characterized by the abnormal accumulation of aggregates of alpha-synuclein protein in neurons, nerve fibres or glial cells. [url:http://en.wikipedia.org/wiki/Synucleinopathies ] ['MESH:D000080874' 'MONDO:0000510' 'UMLS:C5191670' 'MEDGEN:1682194'] ['EFO_0000618' 'OTAR_0000018' 'OTAR_0000020'] ['MONDO_0019052' 'MONDO_0021179' 'MONDO_0024237'] Unknown_Phenotypes"

## Target DF

In [123]:
target_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'target'),columns=['id', 'approvedSymbol', 'biotype', #'transcriptIds', "canonicalExons",
       'genomicLocation', 'alternativeGenes', 'approvedName', 'go', 'hallmarks', 'synonyms',
       'functionDescriptions',  'subcellularLocations', 'targetClass', 'constraint', 'tep', 'proteinIds',# 'dbXrefs','chemicalProbes',
            # 'homologues', # useful and interesting but needs processing before use as feature - skip for now
         'tractability', 'safetyLiabilities',
       'pathways',# 'tss'
        ]).dropna(how="all",axis=1)#.rename(columns={"id":"targetId"}) # do rename later ,for compat with target pipeline code
print(target_df.shape[0])
# target_df = target_df.loc[target_df["targetId"].isin(valid_targets)]
print(target_df.shape[0])
assert target_df.shape[0]>0

# make into list of the go terms
target_df["go"] = target_df["go"].map(
    lambda x: list(dict.fromkeys(d["id"] for d in x if isinstance(d, dict) and "id" in d))
             if isinstance(x, (list, tuple, np.ndarray)) else []
)

target_df["proteinIds"] = target_df["proteinIds"].map(
    lambda x: list(dict.fromkeys(d["id"] for d in x if isinstance(d, dict) and "id" in d))
             if isinstance(x, (list, tuple, np.ndarray)) else []
)

target_df["synonyms"] = target_df["synonyms"].str.lower().map(
    lambda x: list(dict.fromkeys(d["label"] for d in x if isinstance(d, dict) and "label" in d))
             if isinstance(x, (list, tuple, np.ndarray)) else [])
target_df["count_synonyms"] = target_df["synonyms"].str.len().fillna(0)
target_df["synonyms"] = target_df["synonyms"].str.slice(0,10) # first 10

target_df["count_subcellularLocations"] = target_df["subcellularLocations"].str.len().fillna(0) # before splitting
target_df["subcellularLocations"] = target_df["subcellularLocations"].map(
    lambda x:list(dict.fromkeys(d["location"] for d in x if isinstance(d, dict) and "location" in d)
             if isinstance(x, (list, tuple, np.ndarray)) else [])
)
# target_df["subcellularLocations"] = target_df["subcellularLocations"].str.join(", ")

# ## remove the "false" values, make it easier to extract
# target_df['tractability'] = target_df['tractability'].apply(
#     lambda v: list(dict.fromkeys([d for d in (v if isinstance(v, (list, tuple, np.ndarray)) else [v]]))
#                if isinstance(d, dict) and d.get('value', False))
# )
target_df["tractability"] = target_df["tractability"].apply(
    lambda v: list({                               # dedupe, keep insertion order (Py 3.7+)
        frozenset(d.items()): d                    # hashable surrogate key → original dict
        for d in (v if isinstance(v, (list, tuple, np.ndarray)) else [v])
        if isinstance(d, dict) and d.get("value")  # keep only truthy-value dicts
    }.values())
)
## get values from tractability. "value" seems to be only true or false. get vals for less text:

target_df["tractability"] = target_df["tractability"].map(
    lambda x: list((" ".join([d["modality"],d["id"]]) for d in x if isinstance(d, dict) and "id" in d))
             if isinstance(x, (list, tuple, np.ndarray)) else [])

## make into string from list:
# target_df["go_terms_str"] = target_df["go_terms"].str.join(",")

## add some count features of lists of terms. NOTE: doesn't distinguish on values, or falses etc 
for c in ["go","pathways","proteinIds","targetClass","safetyLiabilities","tractability",
          "alternativeGenes","hallmarks","functionDescriptions","tep"]: # "homologues",
    if c in target_df.columns:
        target_df[f"count_{c}"] = target_df[c].str.len().fillna(0).astype(int)
        print(c,"max len",target_df[f"count_{c}"].max())
        try:
            ## truncate length! Also may ruin string format!
            # target_df[c] = target_df[c].str.slice(0,500) # ORIG char based
            target_df[c] = target_df[c].apply(lambda x: " ".join(x.split()[:2_000]) if isinstance(x, str) else x[:250] if isinstance(x, list) else x)
        except:
            print(c,"ERROR!")

target_df["functionDescriptions"] =  target_df["functionDescriptions"].map(lambda x: " ".join(x),na_action="ignore").fillna("").str.strip() # as str

target_df["synonyms"] = target_df["synonyms"].apply(" ".join).str.strip() # make str. 
#Add joint text description col, used for embeddings.
## 'id', 'approvedSymbol', - not use if using for featurees vs embed
## symbol without numbers - pseudo gene family
target_df["sym"] = target_df['approvedSymbol'].str.replace(r'\d+', '', regex=True)
# ##ORIG:
# target_df["target_text_embed"] = target_df[["sym","approvedName","synonyms","functionDescriptions"]].astype(str).apply(" ".join,axis=1)
##ALT: Expanded text
target_df["target_text_embed"] = target_df[["sym","approvedName","synonyms","functionDescriptions","go","tractability","pathways",
                                             'targetClass']].astype(str).apply(" ".join,axis=1) # , 'constraint' : numbers
def extract_constraint_scores(row, key):
    """
    Get constraint scores
    >
    target_df['score_syn'] = target_df['constraint'].apply(lambda x: extract_scores(x, 'syn'))
    target_df['score_mis'] = target_df['constraint'].apply(lambda x: extract_scores(x, 'mis'))
    target_df['score_lof'] = target_df['constraint'].apply(lambda x: extract_scores(x, 'lof'))
    """
    if isinstance(row, (list, np.ndarray)):
        for item in row:
            if isinstance(item, dict) and item.get('constraintType') == key:
                return item.get('score')
    return np.nan

# Apply to create new columns
target_df['score_syn'] = target_df['constraint'].apply(lambda x: extract_constraint_scores(x, 'syn'))
target_df['score_mis'] = target_df['constraint'].apply(lambda x: extract_constraint_scores(x, 'mis'))
target_df['score_lof'] = target_df['constraint'].apply(lambda x: extract_constraint_scores(x, 'lof'))

# target_df["hasSafetyLiabilities"] = target_df["safetyLiabilities"].isna().astype(int)
# target_df["Nohallmarks"] = target_df["hallmarks"].isna().astype(int)
print(target_df["id"].nunique(),"# unique targets")
display(target_df.head(3))

78726
78726
go max len 284
pathways max len 205
proteinIds max len 272
targetClass max len 153
safetyLiabilities max len 79
tractability max len 17
alternativeGenes max len 23
hallmarks max len 2
functionDescriptions max len 20
tep max len 4
78726 # unique targets


,id,approvedSymbol,biotype,genomicLocation,alternativeGenes,approvedName,go,hallmarks,synonyms,functionDescriptions,...,count_tractability,count_alternativeGenes,count_hallmarks,count_functionDescriptions,count_tep,sym,target_text_embed,score_syn,score_mis,score_lof
0,ENSG00000000457,SCYL3,protein_coding,"{'chromosome': '1', 'start': 169849631, 'end':...",None,SCY1 like pseudokinase 3,"[GO:0005737, GO:0042802, GO:0016477, GO:000551...",None,,May play a role in regulating cell adhesion/mi...,...,1,0,0,1,0,SCYL,SCYL SCY1 like pseudokinase 3 May play a role...,0.70818,0.98492,2.815100e-01
1,ENSG00000001167,NFYA,protein_coding,"{'chromosome': '6', 'start': 41072974, 'end': ...",None,nuclear transcription factor Y subunit alpha,"[GO:0000785, GO:0005515, GO:0016602, GO:000635...",None,,Component of the sequence-specific heterotrime...,...,2,0,0,1,0,NFYA,NFYA nuclear transcription factor Y subunit al...,-0.17463,2.78030,1.461900e-01
2,ENSG00000001460,STPG1,protein_coding,"{'chromosome': '1', 'start': 24356999, 'end': ...",None,sperm tail PG-rich repeat containing 1,"[GO:1902110, GO:0005634, GO:0005737, GO:000691...",None,,May positively contribute to the induction of ...,...,0,0,0,1,0,STPG,STPG sperm tail PG-rich repeat containing 1 M...,0.54788,0.24957,1.204300e-09


In [124]:
# target_df["target_text_embed"].str.split().str.len().describe().round(1)

In [125]:
target_df[target_df["tractability"].astype(str).str.contains("drug",case=False,na=False)]["id"].nunique()

2602

In [126]:
target_df[target_df["count_tractability"]>0]["id"].nunique()

17047

In [127]:
# target_df.drop_duplicates("hallmarks")["hallmarks"].values
target_df["hallmarks"].isna().mean() ## 99% nan/missing

np.float64(0.9953890709549577)

In [128]:
target_df["tractability"].head().values

array([list(['PR Database Ubiquitination']),
       list(['PR Database Ubiquitination', 'PR Half-life Data']),
       list([]),
       list(['AB Human Protein Atlas loc', 'PR Database Ubiquitination', 'PR Half-life Data']),
       list(['SM Druggable Family', 'PR Database Ubiquitination'])],
      dtype=object)

#### mini filter of targets for tractability
* Could use 2017 list of around 2.5~4.5 K.
* For now, keep those with any tractability (no filter on quality) or known drug (1522) , leaving around 17K out of 78 K.
We will want the "Druggable genome" for candidates. (But not for Retrieval model training or targets)

In [129]:
known_drug_ids = pd.read_parquet(os.path.join(Config.DATA_DIR, 'known_drug'),columns=["targetId"])["targetId"].unique() # 1522
print(target_df["id"].nunique()) 
target_df = target_df.loc[(target_df["count_tractability"]>0)|(target_df["id"].isin(known_drug_ids))]
print(target_df["id"].nunique(),"After tractability or known drug filterin")

78726
17065 After tractability or known drug filterin


In [130]:
# known_drug_df["targetId"].nunique() # 1522

In [131]:
target_df["target_text_embed"].str.split().str.len().describe().round(1)

count    17065.0
mean       146.5
std        147.9
min          8.0
25%         54.0
50%        104.0
75%        188.0
max       2732.0
Name: target_text_embed, dtype: float64

In [132]:
disease_df

,id,name,description,dbXRefs,synonyms,ancestors,parents,therapeuticAreas,ExactSynonyms,disease_text_embed
0,DOID_0050890,synucleinopathy,A neurodegenerative disease that is characteri...,"[MESH:D000080874, MONDO:0000510, UMLS:C5191670...",{'hasExactSynonym': ['alpha Synucleinopathies'...,"[MONDO_0024237, EFO_0005772, EFO_0000618, MOND...","[MONDO_0019052, MONDO_0021179, MONDO_0024237]","[EFO_0000618, OTAR_0000018, OTAR_0000020]",alpha Synucleinopathies synucleinopathy,synucleinopathy alpha Synucleinopathies synucl...
1,DOID_10113,trypanosomiasis,Infection with protozoa of the genus trypanosoma.,"[UMLS:C0041227, MONDO:0000940, ICD10CM:B56, Me...",{'hasExactSynonym': ['Trypanosoma caused disea...,"[MONDO_0002428, EFO_0001067, EFO_0005741]",[MONDO_0002428],[EFO_0005741],Trypanosoma caused disease or disorder Trypano...,trypanosomiasis Trypanosoma caused disease or ...
2,DOID_10718,giardiasis,An infection of the small intestine caused by ...,"[MeSH:D005873, ICD9CM:007.1, MESH:D005873, MON...","{'hasExactSynonym': ['giardiasis', 'beaver fea...","[EFO_0010282, EFO_0009431, EFO_0009561, EFO_00...","[MONDO_0002428, EFO_0009561]","[EFO_0010282, EFO_0005741]",giardiasis beaver feaver beaver fever infectio...,giardiasis giardiasis beaver feaver beaver fev...
3,DOID_13406,pulmonary sarcoidosis,Sarcoidosis affecting the lung parenchyma. It ...,"[SNOMEDCT:187230004, UMLS:C0036205, MONDO:0001...","{'hasExactSynonym': ['Sarcoidosis, Pulmonary',...","[MONDO_0017026, MONDO_0019338, EFO_0004244, EF...","[MONDO_0017026, MONDO_0019338]","[OTAR_0000010, OTAR_0000006]","Sarcoidosis, Pulmonary sarcoidosis of lung lun...","pulmonary sarcoidosis Sarcoidosis, Pulmonary s..."
4,DOID_1947,trichomoniasis,An infection that is caused by Trichomonas.,"[ICD9:131.8, MESH:D014245, ICD10CM:A59, ICD9CM...","{'hasExactSynonym': ['trichomoniasis', 'Tricho...","[EFO_0005741, EFO_0001067, MONDO_0002428]",[MONDO_0002428],[EFO_0005741],trichomoniasis Trichomonas Infections Trichomo...,trichomoniasis trichomoniasis Trichomonas Infe...
...,...,...,...,...,...,...,...,...,...,...
38954,Orphanet_99942,Autosomal dominant Charcot-Marie-Tooth disease...,Autosomal dominant Charcot-Marie-Tooth disease...,"[ICD10:G60.0, OMIM:607677]","{'hasExactSynonym': ['CMT2I'], 'hasRelatedSyno...","[OTAR_0000018, EFO_0003100, EFO_0000618, MONDO...",[MONDO_0018993],"[OTAR_0000018, EFO_0000618]",CMT2I,Autosomal dominant Charcot-Marie-Tooth disease...
38955,Orphanet_99943,Autosomal dominant Charcot-Marie-Tooth disease...,Autosomal dominant Charcot-Marie-Tooth disease...,"[ICD10:G60.0, OMIM:607736]","{'hasExactSynonym': ['CMT2J'], 'hasRelatedSyno...","[EFO_1001902, EFO_0009387, EFO_0004149, MONDO_...",[MONDO_0018993],"[EFO_0000618, OTAR_0000018]",CMT2J,Autosomal dominant Charcot-Marie-Tooth disease...
38956,Orphanet_99945,Autosomal dominant Charcot-Marie-Tooth disease...,Autosomal dominant Charcot-Marie-Tooth disease...,"[ICD10:G60.0, OMIM:608673]","{'hasExactSynonym': ['CMT2L'], 'hasRelatedSyno...","[EFO_1001902, EFO_0003100, EFO_0000508, MONDO_...",[MONDO_0018993],"[OTAR_0000018, EFO_0000618]",CMT2L,Autosomal dominant Charcot-Marie-Tooth disease...
38957,Orphanet_99946,Autosomal dominant Charcot-Marie-Tooth disease...,Autosomal dominant Charcot-Marie-Tooth disease...,"[ICD10:G60.0, OMIM:118210]","{'hasExactSynonym': ['CMT2A1'], 'hasRelatedSyn...","[MONDO_0100546, MONDO_0020127, EFO_0003100, EF...",[MONDO_0018993],"[EFO_0000618, OTAR_0000018]",CMT2A1,Autosomal dominant Charcot-Marie-Tooth disease...


In [133]:
# 01_prepare_data.py
# GOAL: To perform all heavy, one-time data processing and feature engineering.
# This script is independent of the modeling library and produces clean, ML-ready files.
import os
import numpy as np
import pandas as pd

class Config:
    DATA_DIR = "../data/opentargets/"
    PROCESSED_DATA_FILE = 'final_df.parquet'
    DISEASE_EMBEDDINGS_FILE = 'disease_embeddings.npz'
    TARGET_EMBEDDINGS_FILE = 'target_embeddings.npz'
    # TEXT_MODEL = 'sentence-transformers/all-MiniLM-L12-v2' # originally used 
    TEXT_MODEL = 'thomas-sounack/BioClinical-ModernBERT-large'
    # TEXT_MODEL =  "nomic-ai/modernbert-embed-base" "lightonai/modernbert-embed-large" # Needs PREFIXES!! https://huggingface.co/lightonai/modernbert-embed-large 
    # https://huggingface.co/thomas-sounack/BioClinical-ModernBERT-base
# biolord; lokeshch19/ModernPubMedBERT 


### newly moved up here,outside the "main":
# ## Always run this section; so we can have its filtered diseases in data (for filtering candidate diseases):
### following section extracted; used to filter out diseases (e.g. "measurement")
ta_map = disease_df.set_index('id')['name'].to_dict()
labels_to_remove = ['measurement', 'phenotype', 'biological process', 'cell proliferation disorder']
exploded_tas = disease_df[['id', 'therapeuticAreas']].explode('therapeuticAreas').dropna()
exploded_tas['label'] = exploded_tas['therapeuticAreas'].map(ta_map)
ids_to_remove = exploded_tas[exploded_tas['label'].isin(labels_to_remove)]['id'].unique()

def make_target_data():
    """Orchestrates the data processing and feature generation pipeline."""
    print("--- Starting Data and Feature Pre-computation ---")
    
    if not os.path.exists(Config.PROCESSED_DATA_FILE):
        print(f"-> '{Config.PROCESSED_DATA_FILE}' not found. Processing raw data...")
        # disease_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'disease'))
        associations_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'association_overall_direct'))
        known_drug_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'known_drug'))
        
        associations_filtered = associations_df[~associations_df['diseaseId'].isin(ids_to_remove)]
        ###### Should we be dropping phase - na cases? (dropna)?? (Also later on) ######
        validated_targets = known_drug_df.dropna(subset=['phase'])['targetId'].unique()
        working_df = associations_filtered[associations_filtered['targetId'].isin(validated_targets)].copy()
        
        pairs_with_evidence = known_drug_df.dropna(subset=['phase'])[['targetId', 'diseaseId']].drop_duplicates()
        pairs_with_evidence['label'] = 1
        
        final_df = pd.merge(working_df, pairs_with_evidence, on=['targetId', 'diseaseId'], how='left')
        final_df['label'] = final_df['label'].fillna(0).astype(int)
        print(final_df['label'].describe().round(3))
        final_df.to_parquet(Config.PROCESSED_DATA_FILE)
        print(f"-> Saved processed data to '{Config.PROCESSED_DATA_FILE}'.")
    else:
        print(f"-> Found '{Config.PROCESSED_DATA_FILE}'. Skipping raw data processing.")
def get_pretrained_text_embeddings():
    ## could filter by IDs in the training dataset? 
    #### note: normalize_embeddings may GREATLY reduce perf (from 96 to 90) ?
    if not os.path.exists(Config.DISEASE_EMBEDDINGS_FILE):
        from sentence_transformers import SentenceTransformer
        print("-> DISEASE Text embeddings not found. Generating now...")
        model = SentenceTransformer(Config.TEXT_MODEL, model_kwargs={"dtype": "bfloat16"})
        
        # disease_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'disease'))
        # target_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'target'))
        
        unique_diseases = disease_df[['id', 'disease_text_embed']].dropna().drop_duplicates('id').set_index('id')
        disease_embeddings = model.encode(unique_diseases['disease_text_embed'].tolist(), show_progress_bar=True, device='cuda',normalize_embeddings=False)
        # disease_embeddings = model.encode(("search_query: " + unique_diseases['disease_text_embed']).tolist(), show_progress_bar=True, device='cuda',normalize_embeddings=False) # if a model that needs prefixes
        np.savez_compressed(Config.DISEASE_EMBEDDINGS_FILE, ids=unique_diseases.index.values, embeddings=disease_embeddings)
    if not os.path.exists(Config.TARGET_EMBEDDINGS_FILE):
        from sentence_transformers import SentenceTransformer
        print("-> TARGET Text embeddings not found. Generating now...")
        model = SentenceTransformer(Config.TEXT_MODEL, model_kwargs={"dtype": "bfloat16"})
        # unique_targets = target_df[['id', 'approvedSymbol',"approvedName"]].dropna().drop_duplicates('id').set_index('id')
        # target_embeddings = model.encode((unique_targets['approvedSymbol']+" - "+unique_targets["approvedName"]).tolist(), show_progress_bar=True, device='cuda')
        unique_targets = target_df[['id', 'target_text_embed']].dropna().drop_duplicates('id').set_index('id')
        target_embeddings = model.encode((unique_targets['target_text_embed']).tolist(), show_progress_bar=True, device='cuda',normalize_embeddings=False,)
        # target_embeddings = model.encode((("search_document: " + unique_targets['target_text_embed']).tolist()), show_progress_bar=True, device='cuda',normalize_embeddings=False,) # if a model that needs prefixes
        np.savez_compressed(Config.TARGET_EMBEDDINGS_FILE, ids=unique_targets.index.values, embeddings=target_embeddings)
        print("-> Saved text embeddings.")
    else:
        print("-> Found text embeddings. Skipping generation.")
        
    print("\n--- Data and Feature Preparation Complete ---")

if __name__ == "__main__":
    make_target_data()
    get_pretrained_text_embeddings()

--- Starting Data and Feature Pre-computation ---
-> Found 'final_df.parquet'. Skipping raw data processing.
-> Found text embeddings. Skipping generation.

--- Data and Feature Preparation Complete ---


In [134]:
try:
    tractability_all_targets_list = target_df[target_df["count_tractability"]>0]["id"].unique()
    tractability_drug_targets_list =target_df[target_df["tractability"].astype(str).str.contains("drug",case=False,na=False)]["id"].unique()
    
    print(len([c for c in validated_targets if c not in tractability_drug_targets_list])) # 275 , out of 1522
    print(len([c for c in validated_targets if c not in tractability_all_targets_list])) # 18 , out of 1522
except:()

In [135]:
target_df.rename(columns={"id":"targetId"},inplace=True)
target_df["targetId"] = target_df.targetId.astype("category")
disease_df.rename(columns={"id":"diseaseId"},inplace=True)
disease_df["targetId"] = disease_df.diseaseId.astype("category")

In [136]:
try: final_df.shape # see if variable not defined
except:
    final_df = pd.read_parquet(Config.PROCESSED_DATA_FILE)
df_int = final_df.drop(columns=["evidenceCount"],errors="ignore") # score may be nice proxy target, later
df_int.diseaseId = df_int.diseaseId.astype("category")
df_int.targetId = df_int.targetId.astype("category")
unique_diseases= df_int.diseaseId.unique()
# unique_targets = df_int.targetId.unique() # old
print(df_int.nunique())
print(df_int[["score","label"]].mean().round(3))
df_int

diseaseId     12337
targetId       1522
score        172333
label             2
dtype: int64
score    0.049
label    0.102
dtype: float64


,diseaseId,targetId,score,label
0,DOID_0050890,ENSG00000004948,0.002957,0
1,DOID_0050890,ENSG00000005381,0.002217,0
2,DOID_0050890,ENSG00000006210,0.004507,0
3,DOID_0050890,ENSG00000007171,0.009979,0
4,DOID_0050890,ENSG00000007952,0.008685,0
...,...,...,...,...
663346,Orphanet_99947,ENSG00000196569,0.042916,0
663347,Orphanet_99947,ENSG00000196876,0.059107,0
663348,UBERON_0000104,ENSG00000135744,0.002217,0
663349,UBERON_0000104,ENSG00000142168,0.001478,0


### join/union targets (druggable and our data)
* Druggable should theoretically include our data, but whatever, mismatch isnot' massive
* Note that we need to save this if using elsewhere

In [137]:
print(len(unique_targets))
unique_targets = list(set(final_df.targetId.unique()) | set(unique_targets))
print(len(unique_targets))

assert len([c for c in df_int.targetId.astype(str) if c not in unique_targets])==0 
del final_df

4479
4765


In [138]:
if FAST_RUN:
    df_int = df_int.sample(5_000).reset_index(drop=True)

In [139]:
df_int.score.mean()

np.float64(0.049210540936323814)

### OT Score metafeatures
* Features of the OT scores for candidates
* Some are leaks, e.g. if known drugs are known for target-disease
* We can use this so subset and find insights into our candidates/predictions.
    * e.g.: predictions or casess with low genetic/gwas evidence
    * e.g. etiology for treatable targets
    * ...

Features need to be subset. can also include indirect. 

In [140]:
df_assoc_type = pd.read_parquet(os.path.join(Config.DATA_DIR, 'association_by_datatype_direct'))
print(df_assoc_type["datatypeId"].value_counts())
display(df_assoc_type)

df_assoc_direct = pd.read_parquet(os.path.join(Config.DATA_DIR, 'association_overall_direct'))
df_assoc_direct["datatypeId"] = "direct" # pseudo val
display(df_assoc_direct)

datatypeId
literature             2373042
animal_model            662277
genetic_association     600062
rna_expression          160057
somatic_mutation         86406
known_drug               74187
affected_pathway         37820
Name: count, dtype: int64


,diseaseId,targetId,datatypeId,score,evidenceCount
0,DOID_0050890,ENSG00000001084,literature,0.261537,4
1,DOID_0050890,ENSG00000004142,literature,0.018238,1
2,DOID_0050890,ENSG00000004478,literature,0.018238,1
3,DOID_0050890,ENSG00000004948,literature,0.024317,1
4,DOID_0050890,ENSG00000005381,literature,0.018238,1
...,...,...,...,...,...
3993846,Orphanet_99947,ENSG00000284770,animal_model,0.387267,2
3993847,UBERON_0000104,ENSG00000130203,literature,0.018238,1
3993848,UBERON_0000104,ENSG00000135744,literature,0.018238,1
3993849,UBERON_0000104,ENSG00000142168,literature,0.012159,1


,diseaseId,targetId,score,evidenceCount,datatypeId
0,DOID_0050890,ENSG00000001084,0.031799,4,direct
1,DOID_0050890,ENSG00000004142,0.002217,1,direct
2,DOID_0050890,ENSG00000004478,0.002217,1,direct
3,DOID_0050890,ENSG00000004948,0.002957,1,direct
4,DOID_0050890,ENSG00000005381,0.002217,1,direct
...,...,...,...,...,...
3821731,Orphanet_99947,ENSG00000284770,0.047086,2,direct
3821732,UBERON_0000104,ENSG00000130203,0.002217,1,direct
3821733,UBERON_0000104,ENSG00000135744,0.002217,1,direct
3821734,UBERON_0000104,ENSG00000142168,0.001478,1,direct


In [141]:
print("Distribution for known drugs:")
print("known drug - drug-evidence scores")
df_assoc_type_known_drug = df_assoc_type.query('datatypeId=="known_drug"')
print(df_assoc_type_known_drug.describe().round(2))
print("overall evidence scores - known drug (filtered)")
df_assoc_direct.merge(df_assoc_type_known_drug[["diseaseId","targetId"]],on=["diseaseId","targetId"],how="inner")["score"].describe().round(3)

Distribution for known drugs:
known drug - drug-evidence scores
          score  evidenceCount
count  74187.00       74187.00
mean       0.27           7.73
std        0.25          45.44
min        0.02           1.00
25%        0.12           1.00
50%        0.15           2.00
75%        0.45           3.00
max        1.00        2530.00
overall evidence scores - known drug (filtered)


count    74187.000
mean         0.180
std          0.157
min          0.009
25%          0.074
50%          0.092
75%          0.288
max          0.913
Name: score, dtype: float64

In [142]:
df_assoc_type.query('datatypeId=="known_drug"').sort_values("score").head()

,diseaseId,targetId,datatypeId,score,evidenceCount
2152911,HP_0000842,ENSG00000169418,known_drug,0.015198,1
2486955,MONDO_0002050,ENSG00000006283,known_drug,0.015198,1
2487649,MONDO_0002050,ENSG00000100346,known_drug,0.015198,1
2491340,MONDO_0002050,ENSG00000196557,known_drug,0.015198,1
3767216,MONDO_0100342,ENSG00000159131,known_drug,0.015198,1


In [143]:
df_assoc_direct = pd.concat([df_assoc_direct,df_assoc_type.drop(columns=["evidenceCount"])])
# df_assoc_direct

In [144]:
df_assoc_direct = reset_pivoted_multiindex(df_assoc_direct.pivot(index=["diseaseId","targetId"], columns='datatypeId',values=["score"]))
df_assoc_direct

,diseaseId,targetId,score_affected_pathway,score_animal_model,score_direct,score_genetic_association,score_known_drug,score_literature,score_rna_expression,score_somatic_mutation
0,DOID_0050890,ENSG00000001084,NaN,NaN,0.031799,NaN,NaN,0.261537,NaN,NaN
1,DOID_0050890,ENSG00000004142,NaN,NaN,0.002217,NaN,NaN,0.018238,NaN,NaN
2,DOID_0050890,ENSG00000004478,NaN,NaN,0.002217,NaN,NaN,0.018238,NaN,NaN
3,DOID_0050890,ENSG00000004948,NaN,NaN,0.002957,NaN,NaN,0.024317,NaN,NaN
4,DOID_0050890,ENSG00000005381,NaN,NaN,0.002217,NaN,NaN,0.018238,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
3821731,Orphanet_99947,ENSG00000284770,NaN,0.387267,0.047086,NaN,NaN,NaN,NaN,NaN
3821732,UBERON_0000104,ENSG00000130203,NaN,NaN,0.002217,NaN,NaN,0.018238,NaN,NaN
3821733,UBERON_0000104,ENSG00000135744,NaN,NaN,0.002217,NaN,NaN,0.018238,NaN,NaN
3821734,UBERON_0000104,ENSG00000142168,NaN,NaN,0.001478,NaN,NaN,0.012159,NaN,NaN


## Extra features
* need to join +- process
* Easier to add in tree model section

In [145]:
target_priority = pd.read_parquet(os.path.join(DATA_DIR, "target_prioritisation"))
print(target_priority.dropna(subset=["maxClinicalTrialPhase"])["targetId"].nunique())
## drop leaky columns: (+- tractabiltiy later)
target_priority = target_priority.drop(columns=["maxClinicalTrialPhase","hasHighQualityChemicalProbes"],errors="ignore").dropna(axis=0,thresh=2)
target_priority

1555


,targetId,isInMembrane,isSecreted,hasSafetyEvent,hasPocket,hasLigand,hasSmallMoleculeBinder,geneticConstraint,paralogMaxIdentityPercentage,mouseOrthologMaxIdentityPercentage,isCancerDriverGene,hasTEP,mouseKOScore,tissueSpecificity,tissueDistribution
0,ENSG00000000457,0.0,0.0,NaN,0.0,0.0,0.0,-0.598145,0.000000,0.142440,NaN,NaN,0.000000,-1.0,-1.0
1,ENSG00000001167,0.0,0.0,NaN,0.0,0.0,0.0,-0.487914,NaN,0.985590,NaN,NaN,-0.276242,-1.0,-1.0
2,ENSG00000001460,0.0,0.0,NaN,0.0,0.0,0.0,0.487914,0.000000,0.000000,NaN,NaN,NaN,0.5,-1.0
3,ENSG00000001629,1.0,0.0,NaN,0.0,0.0,0.0,-0.880496,0.000000,0.664830,NaN,NaN,-0.166128,-1.0,-1.0
4,ENSG00000003096,0.0,0.0,NaN,0.0,0.0,0.0,-0.355595,-0.726135,0.640065,NaN,NaN,NaN,0.5,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76924,ENSG00000290316,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5,1.0
77001,ENSG00000291237,0.0,0.0,-1.0,1.0,0.0,0.0,NaN,NaN,0.504505,NaN,NaN,-0.973375,0.5,-1.0
77298,ENSG00000295847,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN
77307,ENSG00000295937,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN


#### target_essentiality
* depmap and gene essentiality. Run here due to it being slow - we filter it for our subset of targets first

parsing essentiality is slow + dozens of output cols! 

In [146]:
%%time
# ## slow 
target_essentiality = pd.read_parquet(os.path.join(Config.DATA_DIR, 'target_essentiality'))
print(target_essentiality.shape[0])
target_essentiality = target_essentiality.loc[target_essentiality["id"].isin(unique_targets)] # valid_targets
print(target_essentiality.shape[0])
# target_essentiality = preprocess_open_targets(target_essentiality)
target_essentiality = preprocess_open_targets(df=target_essentiality,top_n_tissues=20) # faster
print(target_essentiality.shape[1])
# target_essentiality = target_essentiality.loc[:,target_essentiality.nunique(dropna=False)!=1] # drop 0 var columns; added into function itself
target_essentiality.rename(columns={"id":"targetId"},inplace=True)
display(target_essentiality) # depmap

17898
4442
30


,targetId,avg_gene_effect,std_gene_effect,min_gene_effect,max_gene_effect,avg_expression,max_expression,n_tissues,n_mutations,n_diseases,...,avg_gene_effect_prostate gland,avg_gene_effect_subdivision of digestive tract,avg_gene_effect_pancreas,avg_gene_effect_lung,avg_gene_effect_external soft tissue zone,avg_gene_effect_craniocervical region,avg_gene_effect_bile duct,avg_gene_effect_Peripheral Nervous System,avg_gene_effect_testis,avg_gene_effect_skin of body
0,ENSG00000003096,-0.011553,0.120418,-0.504687,0.392851,1.752099,8.073607,29,1,77,...,-0.064752,-0.032656,0.041378,-0.004531,-0.013781,-0.030830,-0.007159,-0.034254,-0.027696,0.012177
1,ENSG00000004779,-0.745481,0.301737,-1.733535,0.195903,7.266846,9.490250,29,0,77,...,-0.856242,-0.721680,-0.799968,-0.749492,-0.697183,-0.741716,-0.700790,-0.727939,-0.864704,-0.828448
2,ENSG00000005961,-0.247531,0.115430,-0.785971,0.182921,0.784027,9.391824,29,1,77,...,-0.248261,-0.256792,-0.261391,-0.254471,-0.266412,-0.259292,-0.240389,-0.256261,-0.269487,-0.240035
3,ENSG00000006327,-0.053175,0.133836,-0.761518,0.445257,6.401179,10.447186,29,0,77,...,-0.074804,-0.032117,-0.022908,-0.037546,-0.104551,-0.000699,-0.057827,-0.049577,0.075477,-0.103335
4,ENSG00000007402,-0.121977,0.105092,-0.558649,0.259068,0.744246,6.679480,29,1,77,...,-0.135031,-0.132296,-0.139276,-0.120179,-0.124906,-0.132793,-0.106528,-0.111164,-0.106615,-0.125032
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4437,ENSG00000240344,-0.069803,0.108808,-0.552608,0.445933,5.059563,7.004950,29,1,77,...,-0.015332,-0.083491,-0.078973,-0.063564,-0.037614,-0.067989,-0.062719,-0.089885,0.019198,-0.075610
4438,ENSG00000240403,0.013744,0.155214,-0.788318,0.747874,0.062782,7.446587,29,1,77,...,0.033466,0.002022,0.007458,0.006558,0.011295,0.027143,0.052532,-0.012150,0.130727,-0.019034
4439,ENSG00000241563,-0.022920,0.117131,-0.465262,0.622137,0.350442,4.062640,29,0,77,...,0.027217,-0.028128,-0.038513,0.011073,-0.062593,-0.003535,-0.029460,-0.060792,-0.007783,-0.051927
4440,ENSG00000243646,0.019396,0.091080,-0.673814,0.360581,4.319778,6.802193,29,1,77,...,-0.018569,0.024926,0.017918,0.007791,0.026543,0.012525,0.023998,0.040651,-0.000922,0.021090


CPU times: user 32.9 s, sys: 6.95 s, total: 39.8 s
Wall time: 37.9 s


#### need to aggregate across rows - for mouse and disease phenotype datas

In [147]:
%%time
df_mouse_pheno = pd.read_parquet(DATA_DIR+"mouse_phenotype",columns=['modelPhenotypeClasses', 'modelPhenotypeId',
       'modelPhenotypeLabel', 'targetFromSourceId', #'targetInModel', #  'targetInModelMgiId'
       'targetInModelEnsemblId',])
print(df_mouse_pheno.shape[0])
df_mouse_pheno = df_mouse_pheno.loc[df_mouse_pheno["targetFromSourceId"].isin(unique_targets)]
print(df_mouse_pheno.shape[0],"# rows after filtering by (valid) targetIDs)")
## high level phenotype
df_mouse_pheno["modelPhenotypeClasses"] = df_mouse_pheno["modelPhenotypeClasses"].map(
    lambda x: list(dict.fromkeys(d["label"].replace(" phenotype","") for d in x if isinstance(d, dict) and "id" in d))
             if isinstance(x, (list, tuple, np.ndarray)) else []
)
print(df_mouse_pheno.filter(['modelPhenotypeId', 'modelPhenotypeLabel', 'targetFromSourceId', 'targetInModel',
       'targetInModelEnsemblId', 'targetInModelMgiId']).nunique())
display(df_mouse_pheno)

#### count encoding
counts = df_mouse_pheno['modelPhenotypeId'].value_counts()
threshold = 3  # Example: phenotype/categories appearing 2 or less times are rare
rare_categories = counts[counts <= threshold].index
df_mouse_pheno['modelPhenotypeId'] = df_mouse_pheno['modelPhenotypeId'].replace(rare_categories, 'Rare')


### aggregate into 2 lists
print("Aggregated mouse phenos for targets:")
df_mouse_pheno_agg = df_mouse_pheno.groupby("targetFromSourceId")["modelPhenotypeClasses"].apply(lambda x: list(set([item for sublist in x for item in sublist])),include_groups=False).reset_index()
df_mouse_pheno_agg = df_mouse_pheno_agg.merge(df_mouse_pheno.groupby("targetFromSourceId")["modelPhenotypeId"].agg(lambda x: list(set(x))).reset_index(),
                                             on="targetFromSourceId",how="outer")
df_mouse_pheno_agg.rename(columns={"targetFromSourceId":"targetId"},inplace=True)
df_mouse_pheno_agg.drop_duplicates(subset=["targetId"],inplace=True)
display(df_mouse_pheno_agg)

210579
82413 # rows after filtering by (valid) targetIDs)
modelPhenotypeId          8437
modelPhenotypeLabel       8437
targetFromSourceId        3676
targetInModelEnsemblId    3653
dtype: int64


,modelPhenotypeClasses,modelPhenotypeId,modelPhenotypeLabel,targetFromSourceId,targetInModelEnsemblId
14,[behavior/neurological],MP:0002574,increased vertical activity,ENSG00000115977,ENSMUSG00000057230
19,[respiratory system],MP:0011143,thick lung-associated mesenchyme,ENSG00000167972,ENSMUSG00000024130
20,"[vision/eye, pigmentation]",MP:0005201,abnormal retina pigment epithelium morphology,ENSG00000198691,ENSMUSG00000028125
21,"[vision/eye, nervous system]",MP:0008584,photoreceptor outer segment degeneration,ENSG00000198691,ENSMUSG00000028125
24,[adipose tissue],MP:0001783,decreased white adipose tissue amount,ENSG00000064687,ENSMUSG00000035722
...,...,...,...,...,...
210565,"[hematopoietic system, immune system]",MP:0000689,abnormal spleen morphology,ENSG00000123838,ENSMUSG00000042554
210573,[homeostasis/metabolism],MP:0002118,abnormal lipid homeostasis,ENSG00000101440,ENSMUSG00000027596
210574,[endocrine/exocrine gland],MP:0008296,abnormal adrenal gland x-zone morphology,ENSG00000101440,ENSMUSG00000027596
210575,[cellular],MP:0006038,increased mitochondrial fission,ENSG00000198695,ENSMUSG00000064368


Aggregated mouse phenos for targets:


,targetId,modelPhenotypeClasses,modelPhenotypeId
0,ENSG00000000938,"[limbs/digits/tail, skeleton, cardiovascular s...","[MP:0010124, Rare, MP:0011514, MP:0001194, MP:..."
1,ENSG00000000971,"[nervous system, behavior/neurological, homeos...","[MP:0001554, MP:0006149, MP:0001325, MP:000315..."
2,ENSG00000001084,"[cellular, homeostasis/metabolism, liver/bilia...","[MP:0003890, MP:0002169, MP:0001683, MP:001109..."
3,ENSG00000001617,"[nervous system, behavior/neurological, cellul...","[MP:0001406, MP:0000579, MP:0006009, MP:000106..."
4,ENSG00000001626,"[respiratory system, behavior/neurological, ho...","[MP:0003793, MP:0011085, MP:0000222, MP:000327..."
...,...,...,...
3671,ENSG00000274286,"[respiratory system, homeostasis/metabolism, c...","[MP:0001179, MP:0009674, MP:0001575, Rare, MP:..."
3672,ENSG00000277443,"[nervous system, behavior/neurological, homeos...","[MP:0001325, MP:0002199, MP:0001436, Rare, MP:..."
3673,ENSG00000277893,"[reproductive system, endocrine/exocrine gland...","[MP:0001146, Rare, MP:0002169, MP:0001325, MP:..."
3674,ENSG00000278540,"[growth/size/body region, mortality/aging, emb...","[MP:0011096, MP:0006207, MP:0001730, MP:000398..."


CPU times: user 5.21 s, sys: 19.6 ms, total: 5.23 s
Wall time: 5.24 s


## ABA - developing mouse brain
* map those genes to human genes (partial coverage) and add their features
* list of genes and their features extracted in `aba_download_genelist.ipynb`

In [148]:
df_aba = pd.read_parquet("aba_DevMouseBrain_features.parquet")
# df_aba

In [149]:
## homologues from target_df
df_mouseMap = pd.read_parquet(os.path.join(Config.DATA_DIR, 'target'),columns=['id', 'approvedSymbol','homologues']).dropna(how="all",axis=1)
df_mouseMap.rename(columns={"id":"targetId"},inplace=True)

def first_mouse_dict(v):
    seq = v if isinstance(v, (list, tuple, np.ndarray)) else [v]
    return next((d for d in seq if isinstance(d, dict) and d.get("speciesName") == "Mouse"), None)

df_mouseMap["mouse_homolog"] = df_mouseMap["homologues"].apply(first_mouse_dict)
df_mouseMap.dropna(subset=["mouse_homolog"],axis=0,inplace=True)

df_mouseMap["mouse_homolog_targetGeneId"] = df_mouseMap["mouse_homolog"].map(
    lambda d: d.get("targetGeneId") if isinstance(d, dict) else None
)
df_mouseMap["mouse_homolog_targetGeneSymbol"] = df_mouseMap["mouse_homolog"].map(
    lambda d: d.get("targetGeneSymbol") if isinstance(d, dict) else None
)
df_mouseMap.drop(columns=["homologues","mouse_homolog"],inplace=True, errors="ignore")
# df_mouseMap

In [150]:
len(set(df_aba["gene_acronym"]).intersection(set(df_mouseMap["mouse_homolog_targetGeneSymbol"]))) # 1242

1242

In [151]:
s1 = df_aba.shape[0]
## mouse_homolog_targetGeneId
df_aba = df_aba.merge(df_mouseMap[["targetId","mouse_homolog_targetGeneSymbol"]],left_on="gene_acronym",right_on="mouse_homolog_targetGeneSymbol",how="inner")
print(s1 - df_aba.shape[0],"# aba genes without match")

## for convenience: set target as index + keep only numeric
df_aba = df_aba.set_index("targetId").select_dtypes("number")
display(df_aba)

43 # aba genes without match


,expr_mean,expr_std,expr_max,n_regions,n_ages,time_slope,time_log2fc,peak_age_in_days,auc,auc_norm,...,has_both_sexes,male_time_slope,male_time_log2fc,female_time_log2fc,sex_time_log2fc_diff,F,M,U,_slope_U,_log2fc_U
targetId,,,,,,,,,,,,,,,,,,,,,
ENSG00000129673,0.185388,0.164594,0.624173,11,1,NaN,0.000000,56.0,NaN,NaN,...,0,NaN,0.000000,NaN,NaN,NaN,0.185388,NaN,NaN,NaN
ENSG00000181409,0.410062,0.339406,1.205910,11,1,NaN,0.000000,13.5,NaN,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,0.410062,NaN,0.000000
ENSG00000183044,5.246320,5.437399,12.344700,11,2,0.257614,8.464159,56.0,212.475959,0.501416,...,0,0.257614,8.464159,NaN,NaN,NaN,5.246320,NaN,NaN,NaN
ENSG00000103222,0.140804,0.088643,0.407417,11,2,0.002277,1.638589,56.0,4.387260,0.660585,...,1,NaN,0.000000,0.0,0.0,0.210572,0.126250,0.047934,NaN,0.000000
ENSG00000097007,1.876544,0.627671,2.720340,11,1,NaN,0.000000,56.0,NaN,NaN,...,0,NaN,0.000000,NaN,NaN,NaN,1.876544,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000152977,2.702354,2.975348,16.067900,11,3,0.033620,0.512369,56.0,100.442390,0.756555,...,0,0.033620,0.512369,NaN,NaN,NaN,2.702354,NaN,NaN,NaN
ENSG00000174963,0.111902,0.130402,0.607811,11,8,-0.004959,-1.176532,11.5,3.296355,0.400206,...,0,-0.004335,-1.176532,NaN,NaN,NaN,0.082200,0.201007,-0.093159,-1.447621
ENSG00000139800,2.717903,1.576179,6.298700,11,1,NaN,0.000000,28.0,NaN,NaN,...,0,NaN,0.000000,NaN,NaN,NaN,2.717903,NaN,NaN,NaN


### network, sequence embeddings from String
* preprocessed in `process_string_embed.ipynb` , saved to `"target_string_embed.parquet"`

In [152]:
df_targ_network_embed = pd.read_parquet(os.path.join("../data/","target_string_embed.parquet"))
df_targ_network_embed

,targetId,string_emb_0,string_emb_1,string_emb_2,string_emb_3,string_emb_4,string_emb_5,string_emb_6,string_emb_7,string_emb_8,...,string_emb_502,string_emb_503,string_emb_504,string_emb_505,string_emb_506,string_emb_507,string_emb_508,string_emb_509,string_emb_510,string_emb_511
0,ENSG00000000457,-0.006214,0.038116,-0.042725,0.005058,0.020462,-0.021500,0.002047,0.030136,-0.027359,...,0.038422,-0.020447,0.057526,0.006989,-0.009529,-0.017899,-0.022186,0.025513,0.012001,0.027008
1,ENSG00000001167,-0.011528,-0.030655,0.021393,0.025635,0.028854,-0.002411,0.045502,-0.024536,-0.002871,...,-0.046204,-0.009277,-0.033295,-0.010002,-0.035553,-0.041504,0.006775,0.016449,0.024780,0.003147
2,ENSG00000001460,0.024765,-0.045471,0.018753,0.031647,0.015305,-0.013229,0.048706,0.057465,-0.057159,...,0.016006,0.009933,-0.070129,0.020386,-0.023727,0.136719,0.050568,0.003149,-0.005348,-0.005650
3,ENSG00000001629,0.009300,-0.021088,0.016876,0.039001,0.043640,-0.000889,0.050446,-0.054810,0.016541,...,-0.093994,-0.013931,-0.028259,0.004963,0.010269,0.056885,0.033173,0.004715,0.010040,-0.038757
4,ENSG00000003096,-0.050354,-0.024551,0.010765,0.010414,-0.010353,0.032715,0.030182,0.041138,-0.000097,...,0.015961,0.007317,-0.080566,0.001632,-0.059540,-0.084900,0.064026,-0.023544,-0.010902,-0.005486
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19601,ENSG00000285733,0.032349,0.005112,0.012688,-0.014084,-0.028168,0.005081,0.070007,-0.091858,0.026550,...,-0.046570,-0.007214,0.022018,-0.025864,0.016205,0.052399,0.033112,-0.015556,-0.016144,-0.056244
19602,ENSG00000285880,-0.016129,-0.032379,0.000250,-0.016357,-0.021866,0.001625,-0.052643,0.002066,0.007351,...,0.015610,0.030289,0.036469,-0.017487,-0.005989,-0.010590,-0.052460,0.002460,-0.030746,0.012001
19603,ENSG00000286038,-0.013367,0.040100,0.016251,0.025879,0.014595,-0.012131,-0.010887,0.042297,-0.043365,...,-0.048096,-0.010979,-0.018600,-0.020645,0.055969,0.041931,-0.047729,0.009163,-0.029465,0.072266
19604,ENSG00000286140,0.029648,0.001109,0.009537,-0.044891,-0.067383,-0.002501,0.003275,-0.083984,0.009583,...,0.092407,-0.011932,0.053955,-0.012131,-0.003355,0.151855,0.051208,0.000897,-0.039551,-0.036163


#### down sample (opt) data + subset feature coluns
* add more features later + autocast (keras featureSpace)

In [153]:
disease_df = disease_df.drop_duplicates(subset=["diseaseId"])#.set_index("diseaseId")
target_df = target_df.drop_duplicates(subset=["targetId"])#.set_index("targetId")

In [154]:
df_int

,diseaseId,targetId,score,label
0,DOID_0050890,ENSG00000004948,0.002957,0
1,DOID_0050890,ENSG00000005381,0.002217,0
2,DOID_0050890,ENSG00000006210,0.004507,0
3,DOID_0050890,ENSG00000007171,0.009979,0
4,DOID_0050890,ENSG00000007952,0.008685,0
...,...,...,...,...
663346,Orphanet_99947,ENSG00000196569,0.042916,0
663347,Orphanet_99947,ENSG00000196876,0.059107,0
663348,UBERON_0000104,ENSG00000135744,0.002217,0
663349,UBERON_0000104,ENSG00000142168,0.001478,0


In [155]:
df_learn = df_int#.sample(frac=0.33)#.sample(212_000)
# df_learn.drop(columns=["score"],errors="ignore",inplace=True)
df_learn.label.sum()

df_learn = df_learn.merge(disease_df[["diseaseId","disease_text_embed"]],on="diseaseId")#,left_on="diseaseId",right_index=True)
df_learn = df_learn.merge(target_df[["targetId","target_text_embed"]],on="targetId")#,left_on="targetId",right_index=True)
df_learn.rename(columns={"disease_text_embed":"disease_text","target_text_embed":"target_text"},inplace=True,errors="ignore")
train_df = df_learn
display(train_df)


# ---------------------------------------------------------
# 1)  make a **target–hold-out** split  (70 % / 30 %)
# ---------------------------------------------------------
from sklearn.model_selection import train_test_split

# unique targets → split the *IDs* (not the rows)
train_tids, test_tids = train_test_split(
    df_learn["targetId"].unique(),
    test_size=0.2,          # paper: 70 % targets train, 30 % test
    random_state=42,
    shuffle=True,
    stratify = df_learn.drop_duplicates(subset=["targetId"])["label"]
)

train_df = df_learn[df_learn["targetId"].isin(train_tids)].copy()
test_df  = df_learn[df_learn["targetId"].isin(test_tids )].copy()

# ---------------------------------------------------------
# 2)  optional: split a *validation* set from the train targets
#     (again on targetId, 10 % of the training targets)
# ---------------------------------------------------------
train_tids, val_tids = train_test_split(
    train_tids,
    test_size=0.05,
    random_state=42,
    shuffle=True,
)

val_df   = train_df[train_df["targetId"].isin(val_tids)].copy()
train_df = train_df[train_df["targetId"].isin(train_tids)].copy()

,diseaseId,targetId,score,label,disease_text,target_text
0,DOID_0050890,ENSG00000004948,0.002957,0,synucleinopathy alpha Synucleinopathies synucl...,CALCR calcitonin receptor G protein-coupled r...
1,DOID_0050890,ENSG00000005381,0.002217,0,synucleinopathy alpha Synucleinopathies synucl...,MPO myeloperoxidase Part of the host defense ...
2,DOID_0050890,ENSG00000006210,0.004507,0,synucleinopathy alpha Synucleinopathies synucl...,CXCL C-X3-C motif chemokine ligand 1 Chemokin...
3,DOID_0050890,ENSG00000007171,0.009979,0,synucleinopathy alpha Synucleinopathies synucl...,NOS nitric oxide synthase 2 Produces nitric o...
4,DOID_0050890,ENSG00000007952,0.008685,0,synucleinopathy alpha Synucleinopathies synucl...,NOX NADPH oxidase 1 NADPH oxidase that cataly...
...,...,...,...,...,...,...
663346,Orphanet_99947,ENSG00000196569,0.042916,0,Autosomal dominant Charcot-Marie-Tooth disease...,LAMA laminin subunit alpha 2 Binding to cells...
663347,Orphanet_99947,ENSG00000196876,0.059107,0,Autosomal dominant Charcot-Marie-Tooth disease...,SCNA sodium voltage-gated channel alpha subuni...
663348,UBERON_0000104,ENSG00000135744,0.002217,0,life cycle life entire lifespan entire life cy...,AGT angiotensinogen Essential component of th...
663349,UBERON_0000104,ENSG00000142168,0.001478,0,life cycle life entire lifespan entire life cy...,SOD superoxide dismutase 1 Destroys radicals ...


In [156]:
print("train/val/test target %")
print(train_df["label"].mean().round(2))
print(val_df["label"].mean().round(2))
print(test_df["label"].mean().round(2))

# --- quick sanity checks : that targetIDs are disjoint between train, test, val sets -------------------
assert set(train_tids).isdisjoint(val_tids),  "train & val targets overlap"
assert set(train_tids).isdisjoint(test_tids), "train & test targets overlap"
assert set(val_tids  ).isdisjoint(test_tids), "val & test targets overlap"

# optional: every original row is in exactly one split
assert len(train_df) + len(val_df) + len(test_df) == len(df_learn), \
       "row counts don't add up – some rows were lost or duplicated"

train/val/test target %
0.1
0.08
0.11


In [157]:
df_int.score.mean().round(3)

np.float64(0.049)

In [158]:
## for use in init bias trick - faster convergence to class imbalance ? 
## https://www.tensorflow.org/tutorials/structured_data/imbalanced_data#examine_the_class_label_imbalance
neg, pos = np.bincount(df_learn['label'])
initial_bias = np.log([pos/neg])
initial_bias

array([-2.17733537])

##### minimal sanity check
* Lecks learned id embeddings! 

In [159]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model            import LogisticRegression
from sklearn.metrics                 import roc_auc_score

DO_LR_BASELINE = False 
if DO_LR_BASELINE:
    tfidf = TfidfVectorizer(max_features=2_000, min_df=4)
    Xtr   = tfidf.fit_transform(train_df["disease_text"] + " " + train_df["target_text"])
    Xval  = tfidf.transform(val_df["disease_text"] + " " + val_df ["target_text"])
    Xtest  = tfidf.transform(test_df["disease_text"] + " " + test_df ["target_text"])
    
    clf   = LogisticRegression(max_iter=100, n_jobs=-1).fit(Xtr, train_df["label"])
    print("VAL Sklearn ROC-AUC:", roc_auc_score(val_df["label"], clf.predict_proba(Xval)[:,1]))
    print("Test Skl ROC-AUC:", roc_auc_score(test_df["label"], clf.predict_proba(Xtest)[:,1]))
    
    del Xtr,Xval,Xtest,tfidf

In [160]:
# ### make candidates
## WARNING! Currently missing candidates outsdie of the training data

# # keep exactly one row per target — required by FactorizedTopK
candidates_df = (
    df_learn[["targetId", "target_text"]]      # add other target-side cols later
        .drop_duplicates(subset=["targetId"])  # pandas helper 
        .reset_index(drop=True)
)

In [161]:
if SAVE_INTERMEDIATES:
    ## save these for use in DL model +- catboost model
    ### training data subset - not all candidates!
    df_learn.to_parquet("../data/proc/df_learn.parquet",index=False)
    
    disease_df.to_parquet("../data/proc/disease_df.parquet",index=False)
    target_df.to_parquet("../data/proc/target_df.parquet",index=False)

## DL

* Alt rewrite model (still 2 tower): https://www.tensorflow.org/recommenders/examples/featurization#putting_it_all_together
* Moved and refactored to `1-Train-DL-Retriever.ipynb`

### extra eval on test set

In [162]:
# # ════════════════════════════════════════════════════════════
# # 6.  Retrieval index
# # ════════════════════════════════════════════════════════════
# # cand_vec  = concat([k_fs({"text": candidates_df["target_text"]}),
# #                     targ_emb(targ_lookup(candidates_df["targetId"]))])
# if TRAIN_DL:
#     cand_vec  = concat([k_fs({"text": candidates_df["target_text"]})])
#     cand_embs = k_tower(cand_vec)
    
#     retrieval = BruteForceRetrieval(k=10, return_scores=True)
#     retrieval.update_candidates(cand_embs,
#                                 np.arange(len(candidates_df), dtype="int32"))
    
#     # Example predictions
#     qry_embed = model.encode_q(np.array(["lung fibrosis"]),
#                                np.array(["MONDO_0000160"]))
#     _, idx = retrieval(qry_embed)
#     print("Top‑10 targets:",
#           tf.gather(candidates_df["targetId"].to_numpy(), idx[0]).numpy().tolist())
#     ## reccommend and filter out known positives
#     # -------------------------------------------------------------------
#     # 1.  Build an int→string lookup tensor  (once)
#     # -------------------------------------------------------------------
#     tid_lookup = tf.constant(candidates_df["targetId"].to_numpy())  # shape (N,)
    
#     # ------------- create a set of known (diseaseId, targetId) positives
#     positives = set(zip(df_learn.query("label==1")["diseaseId"],
#                         df_learn.query("label==1")["targetId"]))
#     ######
#     def recommend(disease_id, disease_text, k=7, drop_known=True):
#         q = model.encode_q(np.array([disease_text]), np.array([disease_id]))
#         _, idx = retrieval(q)
#         cand = tf.gather(tid_lookup, idx[0]).numpy().astype(str)
#         if drop_known:
#             cand = [t for t in cand if (disease_id, t) not in positives]
#         return cand[:k]
    
#     # -------------------------------------------------------------------
#     # 2.  Pretty-print an example
#     # -------------------------------------------------------------------
#     targets_dict = target_df.reset_index().set_index("targetId")[["approvedSymbol","approvedName"]].to_dict(orient="index") # map IDs to readable names
    
#     example_id   = "MONDO_0000160"
#     example_text = "lung fibrosis" # needs actual disease text, nvm more features later
    
#     print("Top-10 *novel* targets for", example_text, ":")
#     # print("\n".join(recommend(example_id, example_text, drop_known=True)))
#     res = recommend(example_id, example_text, drop_known=True)
#     print([*map(targets_dict.get, res)])
#     print("\nMONDO_0100233 - long Covid:")
#     # res = recommend("MONDO_0100233", " long haul COVID-19. post-acute sequelae of COVID-19. A chronic disease triggered by acute COVID-19 infection")
#     example_id, example_text = disease_df.loc[disease_df["diseaseId"]=="MONDO_0100233"][["diseaseId","disease_text_embed"]].iloc[0].values
#     res = recommend(example_id, example_text, drop_known=True)
#     # print("\n".join(recommend("MONDO_0100233", " long haul COVID-19. post-acute sequelae of COVID-19. A chronic disease triggered by acute COVID-19 infection")))
#     print([*map(targets_dict.get, res)])

#     print("\n diabetes mellitus type 2 associated cataract")
#     example_id, example_text = disease_df.loc[disease_df["name"]=="diabetes mellitus type 2 associated cataract"][["diseaseId","disease_text_embed"]].iloc[0].values
#     res = recommend(example_id, example_text, drop_known=True)
#     print([*map(targets_dict.get, res)])


#     ## Essential hypertension
#     print("\nEssential hypertension")
#     example_id, example_text = disease_df.loc[disease_df["name"]=="essential hypertension"][["diseaseId","disease_text_embed"]].iloc[0].values
#     res = recommend(example_id, example_text, drop_known=True)
#     print([*map(targets_dict.get, res)])

#     print("\nMultiple sclerosis")
#     example_id, example_text = disease_df.loc[disease_df["name"]=="multiple sclerosis"][["diseaseId","disease_text_embed"]].iloc[0].values
#     res = recommend(example_id, example_text, drop_known=True)
#     print([*map(targets_dict.get, res)])
    
#     print("\nAsperger syndrome/Autism")
#     example_id, example_text = disease_df.loc[disease_df["name"]=="Asperger syndrome"][["diseaseId","disease_text_embed"]].iloc[0].values
#     res = recommend(example_id, example_text, drop_known=True)
#     print([*map(targets_dict.get, res)])    

#     print("\ntreatment refractory schizophrenia") # Schizophrenia which does not respond to common
#     example_id, example_text = disease_df.loc[disease_df["name"]=="treatment refractory schizophrenia"][["diseaseId","disease_text_embed"]].iloc[0].values
#     res = recommend(example_id, example_text, drop_known=True)
#     print([*map(targets_dict.get, res)])    

## Evaluate baselines

In [163]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
)

def evaluate_prob_series(y_true, y_hat, name, threshold=0.5):
    # Handle epsilon vs eps for log_loss
    ll_kwargs = {"labels": [0, 1]}
    param_names = log_loss.__code__.co_varnames
    if "eps" in param_names:
        ll_kwargs["eps"] = 1e-7
    elif "epsilon" in param_names:
        ll_kwargs["epsilon"] = 1e-7

    # Binary predictions from scores
    y_pred_bin = (y_hat >= threshold).astype("int32")

    print(
        f"{name:18s}  "
        f"AUC={roc_auc_score(y_true, y_hat):.4f}  "
        f"PR-AUC={average_precision_score(y_true, y_hat):.4f}  "
        f"BCE={log_loss(y_true, y_hat, **ll_kwargs):.4f}  "
        f"Acc={accuracy_score(y_true, y_pred_bin):.4f}  "
        f"P={precision_score(y_true, y_pred_bin, zero_division=0):.4f}  "
        f"R={recall_score(y_true, y_pred_bin):.4f}"
    )

def target_mean_baseline(test_df, full_df):
    """Return target-wise oracle mean predictions from full dataset (df_learn)."""
    target_means = full_df.groupby("targetId")["label"].mean()
    return test_df["targetId"].map(target_means).fillna(target_means.mean()).to_numpy(dtype="float32")

y_test = test_df["label"].to_numpy(dtype="float32")

evaluate_prob_series(y_test,
                     test_df[["diseaseId"]].merge(train_df.groupby("diseaseId")["label"].mean(),on="diseaseId",how="left")["label"].fillna(0), # mean of disease in train, apply to test
                     "DISEASE mean")

### code for these baselines was deleted
# evaluate_prob_series(y_test,
#                      constant_baseline(test_df, train_df["label"].mean()),
#                      "GLOBAL")

# evaluate_prob_series(y_test,
#                      disease_mean_baseline(test_df, train_df),
#                      "DISEASE mean")

# evaluate_prob_series(
#     y_test,
#     target_mean_baseline(test_df, df_learn),
#     "TARGET mean (oracle)"
# )


DISEASE mean        AUC=0.8703  PR-AUC=0.4824  BCE=0.2823  Acc=0.9005  P=0.7184  R=0.1653


## Make feature dataset for normal tabular learning


In [164]:
df_learn[["diseaseId","targetId"]].nunique()

diseaseId    12337
targetId      1522
dtype: int64

#### save intermediates

In [165]:
if SAVE_INTERMEDIATES:
    # df_learn.to_parquet("./data_sb/merged_OT_v1.snappy.parquet",index=False,compression="snappy") # error: scala.NotImplementedError: Parquet file contains a LIST typed column[6], which is currently unsupported.
    df_learn.to_csv("../data/data_sb/merged_OT_v1.csv.gz",index=False,compression="gzip")
    target_df.loc[target_df["targetId"].isin(df_learn["targetId"])].to_csv("../data/data_sb/target_feats_v1.csv.gz",index=False,compression="gzip")
    
    # target_essentiality.loc[target_essentiality["id"].isin(df_learn["targetId"])].to_parquet("./data_sb/depmap25_v1.snappy.parquet",index=False,compression="snappy")

## OTP overall evidence score baseline

* Our model outperforms the (train on train) baseline of the OT direct evidence score also!
* ~94 auc (cv) vs 91.36 rocauc from the OT score
* PR-AUC: 0.4546

In [166]:
print(df_int.shape[0])
print("OTP Score ROCAUC:", roc_auc_score(df_int["label"], df_int["score"]))
print("OTP Score PR-AUC:", average_precision_score(df_int["label"], df_int["score"]))
## we don't have a cutoff for the score to make it classification trivially
# print("OTP Score classification:\n", classification_report(df["label"], df["score"]))


663351
OTP Score ROCAUC: 0.9136672736423979
OTP Score PR-AUC: 0.4546364881679949


## Load Precomputed text embeddings


In [167]:
import numpy as np
import pandas as pd
# Load the .npy file using numpy.load()
npz = np.load('disease_embeddings.npz',allow_pickle=True)
# Convert the loaded NumPy array to a Pandas DataFrame
df_embed_disease= pd.DataFrame.from_dict({item: npz[item] for item in npz.files}, orient='index').T
df_embed_disease.rename(columns={"ids":"diseaseId","embeddings":"diseaseEmbeddings"},inplace=True)
# df_embed_disease = df_embed_disease.loc[df_embed_disease["diseaseId"].isin(df_learn["diseaseId"])]

# s1 = df_learn.shape[0]
# df_learn = df_learn.merge(df_embed_disease,on="diseaseId",how="left")
# assert s1 == df_learn.shape[0]

npz = np.load('target_embeddings.npz',allow_pickle=True)
# Convert the loaded NumPy array to a Pandas DataFrame
df_embed_target= pd.DataFrame.from_dict({item: npz[item] for item in npz.files}, orient='index').T
df_embed_target.rename(columns={"ids":"targetId","embeddings":"targetEmbeddings"},inplace=True)

# s1 = df_learn.shape[0]
# df_learn = df_learn.merge(df_embed_target,on="targetId",how="left")
# assert s1 == df_learn.shape[0]

In [168]:
df_learn.diseaseId.nunique()

12337

## Local Catboost/Tree model
* Can use embeddings (pretrained) , and get basic text and categorical features
* https://catboost.ai/docs/en/concepts/python-usages-examples#class-with-array-like-data-with-numerical,-categorical-and-embedding-features

* https://chatgpt.com/share/688fd1c2-569c-8013-92ec-d11748791a5c

## Tree model
#### very compute intensive! may hang and takes a while! 
* not sure what is heaviest part
* `embedding_features` (when explicitly fed a such to CB) are very _heavy_!


* text features in catboost example :
    * https://colab.research.google.com/github/catboost/tutorials/blob/master/text_features/text_features_in_catboost.ipynb#scrollTo=4I1cDLrr4T-b
    * CB also supports BPE, dictionaries= ['Word:min_token_occurrence=5']...

In [169]:
%%time
y_train = train_df.label
y_test = test_df.label
y_val = val_df.label

X_train = get_cb_x(train_df)
X_test = get_cb_x(test_df)
X_val = get_cb_x(val_df)

(505221, 579)
(130724, 579)
(27406, 579)
CPU times: user 10.2 s, sys: 3.63 s, total: 13.8 s
Wall time: 13.8 s


In [170]:
X_train.select_dtypes(["number","bool"]).columns.tolist()[0:40]

['count_synonyms',
 'count_subcellularLocations',
 'count_go',
 'count_pathways',
 'count_proteinIds',
 'count_targetClass',
 'count_safetyLiabilities',
 'count_tractability',
 'count_alternativeGenes',
 'count_hallmarks',
 'count_functionDescriptions',
 'count_tep',
 'score_syn',
 'score_mis',
 'score_lof',
 'isInMembrane',
 'isSecreted',
 'hasSafetyEvent',
 'hasPocket',
 'hasLigand',
 'hasSmallMoleculeBinder',
 'geneticConstraint',
 'paralogMaxIdentityPercentage',
 'mouseOrthologMaxIdentityPercentage',
 'isCancerDriverGene',
 'hasTEP',
 'mouseKOScore',
 'tissueSpecificity',
 'tissueDistribution',
 'phenotypes_count',
 'diseaseHasPheno',
 'avg_gene_effect',
 'std_gene_effect',
 'min_gene_effect',
 'max_gene_effect',
 'avg_expression',
 'max_expression',
 'n_tissues',
 'n_mutations',
 'n_diseases']

In [171]:
text_features = [c for c in text_features if c in X_test.columns]
cat_features = [c for c in cat_features if c in X_test.columns]
if embed_cols: #not none
    embed_cols = [c for c in embed_cols if c in X_test.columns]

In [172]:
## if want to add (catboost supported) embedding features (pretrained, target, disease)
# X_train_emb['embeddings'] = X_train.values.tolist() 
# and set embed cols


In [173]:

import gc
try:
    del model # clean dl model
except:()
gc.collect()

0

#### Train CB model

In [174]:
learn_pool = Pool(
    X_train, 
    y_train, 
    cat_features=cat_features,
    text_features=text_features,
    embedding_features=embed_cols,
)

val_pool = Pool(
    X_val, 
    y_val, 
    cat_features=cat_features,
    text_features=text_features,
    embedding_features=embed_cols,
)

test_pool = Pool(
    X_test, 
    y_test, 
    cat_features=cat_features,
    text_features=text_features,
    embedding_features=embed_cols,
)
print("Pools ready")
# model = CatBoostClassifier(**CFG["cb_params"])
model = CatBoostClassifier(iterations=40 if FAST_RUN else 300, depth=5 if FAST_RUN else 8,
                          eval_metric="AUC", early_stopping_rounds=50, 
                          task_type="GPU", ## runs out of mem with many cols
                          random_seed=42,auto_class_weights = "SqrtBalanced",)
model.fit(learn_pool, eval_set=val_pool, verbose=False,plot=True) # , verbose=100

Pools ready


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Default metric period is 5 because AUC is/are not implemented for GPU


In [175]:
test_preds = model.predict(test_pool,prediction_type="Probability")[:,1]
print("rocauc:", round(roc_auc_score(y_true=y_test,y_score=test_preds),4))
print("average prec (pseudo pr-auc?):", round(average_precision_score(y_true=y_test,y_score=test_preds),4))
print(classification_report(y_true=y_test, y_pred=model.predict(test_pool)))

rocauc: 0.9428
average prec (pseudo pr-auc?): 0.7654
              precision    recall  f1-score   support

           0       0.96      0.96      0.96    116258
           1       0.69      0.67      0.68     14466

    accuracy                           0.93    130724
   macro avg       0.82      0.82      0.82    130724
weighted avg       0.93      0.93      0.93    130724



```
rocauc: 0.9398
average prec (pseudo pr-auc?): 0.7573
              precision    recall  f1-score   support

           0       0.96      0.97      0.96    116258
           1       0.70      0.64      0.67     14466

    accuracy                           0.93    130724
   macro avg       0.83      0.80      0.81    130724
weighted avg       0.93      0.93      0.93    130724
```

In [176]:
model.get_feature_importance(data=test_pool,
                       # reference_data=None,
                       # type=EFstrType.FeatureImportance,
                       prettified=True).head(20).round(2)

,Feature Id,Importances
0,target_text,31.21
1,disease_text,22.12
2,diseaseId,9.17
3,ancestors,7.83
4,phenotypes,2.16
5,string_emb_461,1.32
6,string_emb_2,0.87
7,string_emb_440,0.76
8,count_targetClass,0.59
9,count_safetyLiabilities,0.53


In [177]:
model.get_feature_importance(data=val_pool,
                       # reference_data=test_pool,
                       # type="ShapValues",
                       type="SageValues",
                       sage_n_samples=3_048,
                       prettified=True).head(15).round(3)

,Feature Id,Importances
0,target_text,0.257
1,diseaseId,0.132
2,disease_text,0.094
3,ancestors,0.014
4,phenotypes,0.010
5,string_emb_461,0.003
6,string_emb_440,0.003
7,string_emb_153,0.002
8,count_safetyLiabilities,0.002
9,string_emb_2,0.002


#### try to get shap interactions - Slowish

In [178]:
# ## this is interactions of pairs, including linear contributions. 
# pd.DataFrame(model.get_feature_importance(val_pool, type='Interaction'),
#              columns=["i","j","score"]) \
#   .assign(feat_i=lambda d: d["i"].map(dict(enumerate(X_val.columns))),
#           feat_j=lambda d: d["j"].map(dict(enumerate(X_val.columns)))) \
#   [["feat_i","feat_j","score"]] \
#   .sort_values("score", ascending=False) \
#   .head(10).round(2)

* Get the interaction contribution (diagonal on shap), beyond just how much each added individually

In [179]:
%%time
import numpy as np, pandas as pd, shap
## gets stuck on full data! / slow
def top_shap_interactions(model, val_pool, feature_names, top_k=10):
    explainer = shap.TreeExplainer(model)
    S = explainer.shap_interaction_values(val_pool)  # (n, m, m) or (n, m, m, C)

    # Handle multi-class by averaging over classes
    if isinstance(S, np.ndarray) and S.ndim == 4:
        S = S.mean(axis=-1)  # average over classes -> (n, m, m)

    # Global mean |interaction| for each pair
    S_abs = np.abs(S).mean(axis=0)  # (m, m)
    main_abs = np.abs(np.diagonal(S, axis1=1, axis2=2)).mean(axis=0)  # (m,)
    np.fill_diagonal(S_abs, 0.0)

    m = len(feature_names)
    rows = []
    for i in range(m):
        for j in range(i+1, m):
            inter = S_abs[i, j]
            denom = (main_abs[i] + main_abs[j]) + 1e-12
            rows.append((feature_names[i], feature_names[j], inter, inter/denom))

    return (pd.DataFrame(rows, columns=["feat_i","feat_j","interaction_abs_mean","synergy_ratio"])
              .sort_values(["interaction_abs_mean","synergy_ratio"], ascending=False)
              .head(top_k)
              .round(3))

# # # # usage:
# ###  Can run out of memory!! (But useful analysis!)
# top_synergy = top_shap_interactions(model, val_pool, list(X_val.columns), top_k=15)
# top_synergy

CPU times: user 34 μs, sys: 4 μs, total: 38 μs
Wall time: 42.2 μs


 try to check intersection of go terms
* may need to harmonize acronym/text
* no matching go terms, assuming format is same (not sure)

No intersections found , naively

In [180]:
del learn_pool,val_pool,test_pool
del X_train,X_test,X_val

## CV Results
* run CV
* optional: analysis with and without some input ?

```
count    12337.00
mean         5.47
std         26.21
min          0.00
25%          0.00
75%          0.00
max        628.00
Name: num_known_targets, dtype: float64
```

## added repeated cv code:

In [181]:
def run_catboost_repeated_cv(
    df,
    cat_features,
    text_features,
    embed_cols,
    cb_params,
    n_splits=5,
    n_repeats=5,
    group_col="targetId",
    merge_aba=True,
    base_random_state=42,
    return_model=True,
):
    """
    Repeated groupwise CV (e.g. 5x5) with CatBoost.

    - Uses StratifiedGroupKFold splits by `group_col` so targets are disjoint
      between train/val in every fold.
    - Repeats the CV `n_repeats` times with different random seeds.
    - Returns OOF predictions averaged across repeats (per row).

    Returns:
        oof_df      : DataFrame ["pred","label",group_col], pred = mean OOF over repeats
        metrics_df  : per-fold + per-repeat metrics (cols: repeat, fold, roc_auc, prauc, ...)
        summary     : mean of metrics across all folds and repeats
        model       : last fitted model (if return_model=True)
    """
    df = df.copy()
    index_template = df.index.copy()

    all_oof = []        # list of np.array, one per repeat
    all_metrics = []    # list of metrics_df, one per repeat
    last_model = None
    ##added
    text_features = [c for c in text_features if c in df.columns]
    cat_features = [c for c in cat_features if c in df.columns]
    if embed_cols: #not none
        embed_cols = [c for c in embed_cols if c in df.columns]
    
    for rep in range(n_repeats):
        seed = None if base_random_state is None else base_random_state + rep
        print(f"\n=== Repeated CV run {rep+1}/{n_repeats} (seed={seed}) ===")

        oof_df_single, metrics_df_single, summary_single, model_single = run_catboost_cv(
            df=df,
            cat_features=cat_features,
            text_features=text_features,
            embed_cols=embed_cols,
            cb_params=cb_params,
            n_splits=n_splits,
            group_col=group_col,
            merge_aba=merge_aba,
            random_state=seed,
            return_model=True,
        )

        # Ensure OOF preds are aligned to the original df order
        preds_aligned = oof_df_single.loc[index_template, "pred"].to_numpy()
        all_oof.append(preds_aligned)

        metrics_df_single = metrics_df_single.copy()
        metrics_df_single["repeat"] = rep
        all_metrics.append(metrics_df_single)

        last_model = model_single

    # Shape: (n_repeats, n_samples)
    all_oof = np.vstack(all_oof)
    mean_oof = np.nanmean(all_oof, axis=0)  # just in case there were NaNs

    # Build final OOF df: same structure as before, but preds averaged across repeats
    # Build final OOF df: original df + averaged predictions
    oof_df = df.copy()
    oof_df["pred"] = mean_oof

    metrics_df = pd.concat(all_metrics, ignore_index=True)
    summary = metrics_df.mean(numeric_only=True).round(4).to_dict()
    print("Repeated-CV summary:", summary)

    if return_model:
        return oof_df, metrics_df, summary, last_model
    else:
        return oof_df, metrics_df, summary


In [182]:
# train_df[["diseaseId","targetId"]+text_features]

In [183]:
train_df.columns

Index(['diseaseId', 'targetId', 'score', 'label', 'disease_text',
       'target_text'],
      dtype='object')

* new run without added features

In [184]:
%%time
if RUN_CV:
    df_all = pd.concat([train_df, test_df, val_df]).drop_duplicates()#.sample(n=1_000)
    
    oof_pred, fold_metrics, summary, model = run_catboost_repeated_cv(
        df=df_all,
        cat_features=cat_features,
        # text_features=text_features,
        # embed_cols=embed_cols,
        # cat_features=[],
        text_features=text_features,
        embed_cols=embed_cols,
        cb_params=CFG["cb_params"],
        n_splits=5,
        n_repeats=5,          # 5×5 CV
        group_col="targetId", # target-disjoint folds
        merge_aba=False,      # or True, same as you used before
    )
    
    print("Mean metrics summary over 5×5 CV:")
    print({k: round(v, 4) for k, v in summary.items()})
    
    # sanity checks (same as you had)
    assert oof_pred.dropna(axis=0).shape[0] == oof_pred.shape[0]
    assert oof_pred.shape[0] == df_all.shape[0]


=== Repeated CV run 1/5 (seed=42) ===
(663351, 7)
(663351, 3) - X.shape
Index(['diseaseId', 'disease_text', 'target_text'], dtype='object')
Learning rate set to 0.023855


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 340ms	remaining: 5m 40s
200:	total: 6.96s	remaining: 27.7s
400:	total: 12.3s	remaining: 18.4s
600:	total: 17.7s	remaining: 11.8s
800:	total: 23.1s	remaining: 5.75s
999:	total: 28.7s	remaining: 0us
Learning rate set to 0.023898


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 237ms	remaining: 3m 57s
200:	total: 6.6s	remaining: 26.2s
400:	total: 12s	remaining: 18s
600:	total: 17.2s	remaining: 11.4s
800:	total: 22.5s	remaining: 5.6s
999:	total: 27.6s	remaining: 0us
Learning rate set to 0.023882


...

0:	total: 165ms	remaining: 2m 44s
200:	total: 5.89s	remaining: 23.4s
400:	total: 10.8s	remaining: 16.2s
600:	total: 15.6s	remaining: 10.4s
800:	total: 20.5s	remaining: 5.09s
999:	total: 25.3s	remaining: 0us
Learning rate set to 0.023868


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 27.8ms	remaining: 27.8s
200:	total: 5.8s	remaining: 23s
400:	total: 10.9s	remaining: 16.3s
600:	total: 15.9s	remaining: 10.6s
800:	total: 21s	remaining: 5.21s
999:	total: 25.8s	remaining: 0us
Learning rate set to 0.023865


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 225ms	remaining: 3m 44s
200:	total: 5.95s	remaining: 23.7s
400:	total: 10.9s	remaining: 16.3s
600:	total: 16.2s	remaining: 10.8s
800:	total: 21.7s	remaining: 5.39s
999:	total: 27.2s	remaining: 0us
{'fold': 2.0, 'roc_auc': 0.9484, 'prauc': 0.7729, 'accuracy': 0.9385, 'f1': 0.6916, 'precision': 0.7086, 'recall': 0.6763}
Repeated-CV summary: {'fold': 2.0, 'roc_auc': 0.9478, 'prauc': 0.7722, 'accuracy': 0.9389, 'f1': 0.6923, 'precision': 0.7106, 'recall': 0.6759, 'repeat': 2.0}
Mean metrics summary over 5×5 CV:
{'fold': 2.0, 'roc_auc': 0.9478, 'prauc': 0.7722, 'accuracy': 0.9389, 'f1': 0.6923, 'precision': 0.7106, 'recall': 0.6759, 'repeat': 2.0}
CPU times: user 48min 43s, sys: 7min 28s, total: 56min 11s
Wall time: 26min 36s


In [185]:
if RUN_CV:
    display(oof_pred)
    display(fold_metrics)
    print("SD:")
    print(fold_metrics.std().round(5))
    print("Summary")
    display(summary)

    oof_pred.to_parquet("./Outputs/CV_tree/CB_5_cv.parquet")
    fold_metrics.to_csv("./Outputs/CV_tree/CB_5_cv_foldMetrics.csv",index=False)
    # summary.to_csv("./Outputs/CV_tree/CB_5_cv_summary.csv",index=False)

,diseaseId,targetId,score,label,disease_text,target_text,pred
0,DOID_0050890,ENSG00000004948,0.002957,0,synucleinopathy alpha Synucleinopathies synucl...,CALCR calcitonin receptor G protein-coupled r...,0.032532
2,DOID_0050890,ENSG00000006210,0.004507,0,synucleinopathy alpha Synucleinopathies synucl...,CXCL C-X3-C motif chemokine ligand 1 Chemokin...,0.004019
3,DOID_0050890,ENSG00000007171,0.009979,0,synucleinopathy alpha Synucleinopathies synucl...,NOS nitric oxide synthase 2 Produces nitric o...,0.004452
4,DOID_0050890,ENSG00000007952,0.008685,0,synucleinopathy alpha Synucleinopathies synucl...,NOX NADPH oxidase 1 NADPH oxidase that cataly...,0.005626
5,DOID_0050890,ENSG00000010610,0.005862,0,synucleinopathy alpha Synucleinopathies synucl...,CD CD4 molecule Integral membrane glycoprotei...,0.016121
...,...,...,...,...,...,...,...
663326,Orphanet_99946,ENSG00000101986,0.054816,0,Autosomal dominant Charcot-Marie-Tooth disease...,ABCD ATP binding cassette subfamily D member 1...,0.003915
663335,Orphanet_99947,ENSG00000101986,0.033809,0,Autosomal dominant Charcot-Marie-Tooth disease...,ABCD ATP binding cassette subfamily D member 1...,0.002175
663339,Orphanet_99947,ENSG00000129244,0.032190,0,Autosomal dominant Charcot-Marie-Tooth disease...,ATPB ATPase Na+/K+ transporting subunit beta 2...,0.003962
663348,UBERON_0000104,ENSG00000135744,0.002217,0,life cycle life entire lifespan entire life cy...,AGT angiotensinogen Essential component of th...,0.013084


,fold,roc_auc,prauc,accuracy,f1,precision,recall,repeat
0,0,0.955988,0.810546,0.939470,0.723438,0.719742,0.727173,0
1,1,0.950350,0.779060,0.943463,0.696718,0.703943,0.689640,0
2,2,0.940199,0.732179,0.936915,0.651531,0.687457,0.619172,0
3,3,0.951456,0.784461,0.939106,0.713793,0.730945,0.697428,0
4,4,0.942433,0.753867,0.935973,0.673706,0.710996,0.640133,0
5,0,0.960331,0.815380,0.947189,0.729343,0.731902,0.726801,1
6,1,0.952559,0.789503,0.940650,0.706238,0.733403,0.681013,1
7,2,0.942101,0.737438,0.938607,0.667447,0.685371,0.650437,1
8,3,0.937396,0.747586,0.937925,0.664972,0.724783,0.614280,1
9,4,0.950691,0.780910,0.932999,0.705268,0.693873,0.717043,1


SD:
fold         1.44338
roc_auc      0.00592
prauc        0.02319
accuracy     0.00460
f1           0.02126
precision    0.01964
recall       0.03432
repeat       1.44338
dtype: float64
Summary


{'fold': 2.0,
 'roc_auc': 0.9478,
 'prauc': 0.7722,
 'accuracy': 0.9389,
 'f1': 0.6923,
 'precision': 0.7106,
 'recall': 0.6759,
 'repeat': 2.0}

```
SD:

roc_auc      0.00604
prauc        0.02186
accuracy     0.00324
f1           0.02046
precision    0.01856
recall       0.03059
 ```
fold_metrics
 ```
{ 'roc_auc': 0.9581,
 'prauc': 0.8203,
 'accuracy': 0.9474,
 'f1': 0.736,
 'precision': 0.7526,
 'recall': 0.7207,
 }
 ```

### Orig cv code follows: 

In [186]:
# %%time
# df_all = pd.concat([train_df,test_df,val_df]).drop_duplicates()
# oof_pred, fold_metrics, summary,model = run_catboost_cv(
#     df=df_all,
#     cat_features=cat_features,         # e.g. ["diseaseId","biotype"]
# a    text_features=text_features,       # your long list with text columns
#     embed_cols=embed_cols,             # None unless you’re using embeddings
#     cb_params=CFG["cb_params"],        # your CatBoost params dict
#     n_splits=2 if FAST_RUN else 5,                        # or CFG["cv_folds"]
#     group_col="targetId",              # disjoint targets between train/val
#     merge_aba=False, #True
# )

# # print("Per-fold metrics:")
# # display(fold_metrics.round(4))
# print("Mean metrics summary:")
# print({k: round(v, 4) for k, v in summary.items()})

# assert oof_pred.dropna(axis=0).shape[0]==oof_pred.shape[0]
# assert oof_pred.dropna(axis=0).shape[0]==df_all.shape[0]



In [187]:
df = df_all.copy()
df["num_known_targets"] = df.groupby("diseaseId")["label"].transform("sum") # for sorting by cases without known
df["in_aba"] = df["targetId"].isin(df_aba.index).astype(int)
df["pred"] = oof_pred["pred"]
display(df.groupby(["in_aba"])[["pred","label"]].mean().round(3))
df["pred_label"] = (df["pred"]>0.502).astype(int)
df["mistake"] = df["pred_label"]!=df["label"]
display(df["mistake"].agg(["mean","sum"]).round(2)) ## this accuracy seems wrong! maybe proba needs weighting? or get actual preds? 

,pred,label
in_aba,,
0,0.137,0.098
1,0.158,0.112


mean        0.06
sum     39546.00
Name: mistake, dtype: float64

Mean metrics summary:
{'fold': 2.0, 'roc_auc': 0.9432, 'prauc': 0.76, 'accuracy': 0.9409, 'f1': 0.6774, 'precision': 0.7545, 'recall': 0.6158}
### before adding in targ string embeds

#### add in sorting by diseases with least number of known positives

In [188]:
df_cands = df.round(2).query('mistake & label==0 & pred>0.51').sort_values(["pred","score"],ascending=False)#.head(15)
df_cands.head(20)

,diseaseId,targetId,score,label,disease_text,target_text,num_known_targets,in_aba,pred,pred_label,mistake
415892,MONDO_0004976,ENSG00000184752,0.34,0,amyotrophic lateral sclerosis Lou Gehrig's dis...,NDUFA NADH:ubiquinone oxidoreductase subunit A...,145,0,1.00,1,True
255710,EFO_0009606,ENSG00000267855,0.14,0,macular degeneration macula lutea retinal dege...,NDUFA NADH:ubiquinone oxidoreductase subunit A...,85,0,1.00,1,True
579321,MONDO_0019172,ENSG00000119421,0.04,0,aniridia aplasia of iris Aniridia is a congeni...,NDUFA NADH:ubiquinone oxidoreductase subunit A...,78,0,1.00,1,True
415854,MONDO_0004976,ENSG00000164258,0.03,0,amyotrophic lateral sclerosis Lou Gehrig's dis...,NDUFS NADH:ubiquinone oxidoreductase subunit S...,145,0,1.00,1,True
296274,EFO_1000616,ENSG00000119421,0.03,0,Uveal Melanoma uveal melanoma iris melanoma me...,NDUFA NADH:ubiquinone oxidoreductase subunit A...,90,0,1.00,1,True
634238,MONDO_0100234,ENSG00000198786,0.00,0,paroxysmal familial ventricular fibrillation v...,MT-ND mitochondrially encoded NADH:ubiquinone ...,44,0,1.00,1,True
415792,MONDO_0004976,ENSG00000127824,0.75,0,amyotrophic lateral sclerosis Lou Gehrig's dis...,TUBAA tubulin alpha 4a Tubulin is the major c...,145,0,0.99,1,True
81222,EFO_0000712,ENSG00000198763,0.34,0,stroke Cerebral Stroke stroke disorder Acute S...,MT-ND mitochondrially encoded NADH:ubiquinone ...,243,0,0.99,1,True
81228,EFO_0000712,ENSG00000198888,0.34,0,stroke Cerebral Stroke stroke disorder Acute S...,MT-ND mitochondrially encoded NADH:ubiquinone ...,243,0,0.99,1,True
636275,MONDO_0100431,ENSG00000198763,0.33,0,migraine without aura common migraine A migrai...,MT-ND mitochondrially encoded NADH:ubiquinone ...,29,0,0.99,1,True


In [189]:
df.round(2).query('num_known_targets<2 & label==0 & pred>0.51 ').sort_values(["pred","score","diseaseId"],ascending=False)[['diseaseId', 'targetId', 'disease_text',
       'target_text', 'num_known_targets',  'pred', 'pred_label']]#.to_csv("orphan_candidates_v1.csv",index=False)

,diseaseId,targetId,disease_text,target_text,num_known_targets,pred,pred_label
534871,MONDO_0014769,ENSG00000261456,inherited oocyte maturation defect oocyte matu...,TUBB tubulin beta 8 class VIII Tubulin is the...,1,0.96,1
385056,MONDO_0002229,ENSG00000258947,ovarian epithelial tumor epithelial tumour of ...,TUBB tubulin beta 3 class III Tubulin is the ...,0,0.90,1
595514,MONDO_0021067,ENSG00000198695,mediastinal germ cell tumor germ cell tumor of...,MT-ND mitochondrially encoded NADH:ubiquinone ...,0,0.88,1
356897,EFO_1002012,ENSG00000171873,ligament rupture Partial or complete tear of ...,ADRAD adrenoceptor alpha 1D This alpha-adrene...,1,0.87,1
288110,EFO_1000366,ENSG00000177084,Mediastinal Malignant Germ Cell Tumor mediasti...,"POLE DNA polymerase epsilon, catalytic subunit...",0,0.84,1
...,...,...,...,...,...,...,...
370300,MONDO_0001160,ENSG00000169432,dissociative disorder dissociative disease dis...,SCNA sodium voltage-gated channel alpha subuni...,0,0.52,1
364704,MONDO_0000553,ENSG00000119013,uterine corpus endometrial carcinoma endometri...,NDUFB NADH:ubiquinone oxidoreductase subunit B...,0,0.52,1
342525,EFO_1001785,ENSG00000138735,diffuse esophageal spasm diffuse spasm of esop...,PDEA phosphodiesterase 5A Plays a role in sig...,0,0.52,1
317606,EFO_1001092,ENSG00000137869,patellofemoral pain syndrome patellofemoral pa...,CYPA cytochrome P450 family 19 subfamily A mem...,1,0.52,1


In [190]:
# df_cands.head(4000).to_csv("sample_novel_preds_v1.csv")

In [191]:
# df.loc[df["disease_text"].str.contains("Multiple sclerosis",case=False)] ## 253 targets
df.loc[(df["disease_text"].str.contains("multiple sclerosis")) & (df["mistake"]) & (df["pred_label"]>0)].sort_values(["pred","score","diseaseId"],ascending=False).round(2)

,diseaseId,targetId,score,label,disease_text,target_text,num_known_targets,in_aba,pred,pred_label,mistake
227127,EFO_0007405,ENSG00000184752,0.00,0,optic neuritis optic neuritis Optic Neuritis O...,NDUFA NADH:ubiquinone oxidoreductase subunit A...,66,0,0.99,1,True
151989,EFO_0003929,ENSG00000099795,0.00,0,relapsing-remitting multiple sclerosis Multipl...,NDUFB NADH:ubiquinone oxidoreductase subunit B...,129,0,0.99,1,True
152097,EFO_0003929,ENSG00000131495,0.00,0,relapsing-remitting multiple sclerosis Multipl...,NDUFA NADH:ubiquinone oxidoreductase subunit A...,129,0,0.99,1,True
152302,EFO_0003929,ENSG00000198840,0.00,0,relapsing-remitting multiple sclerosis Multipl...,MT-ND mitochondrially encoded NADH:ubiquinone ...,129,0,0.99,1,True
152197,EFO_0003929,ENSG00000165264,0.00,0,relapsing-remitting multiple sclerosis Multipl...,NDUFB NADH:ubiquinone oxidoreductase subunit B...,129,0,0.99,1,True
...,...,...,...,...,...,...,...,...,...,...,...
322890,EFO_1001219,ENSG00000104321,0.05,0,trigeminal neuralgia Trigeminal neuralgia trig...,TRPA transient receptor potential cation chann...,16,0,0.51,1,True
429156,MONDO_0005301,ENSG00000104783,0.01,0,multiple sclerosis A progressive autoimmune d...,KCNN potassium calcium-activated channel subfa...,253,0,0.51,1,True
429263,MONDO_0005301,ENSG00000115705,0.01,0,multiple sclerosis A progressive autoimmune d...,TPO thyroid peroxidase Iodination and couplin...,253,0,0.50,1,True
236876,EFO_0008522,ENSG00000258839,0.01,0,secondary progressive multiple sclerosis secon...,MCR melanocortin 1 receptor Receptor for MSH ...,121,0,0.50,1,True


In [192]:
df.query('mistake & label>0').sort_values("pred").round(1).head(10)

,diseaseId,targetId,score,label,disease_text,target_text,num_known_targets,in_aba,pred,pred_label,mistake
654626,Orphanet_411,ENSG00000185624,0.1,1,Hyperlipoproteinemia type 1 Familial hyperchyl...,PHB prolyl 4-hydroxylase subunit beta This mu...,4,0,0.0,0,True
169823,EFO_0004264,ENSG00000112062,0.1,1,vascular disease vascular tissue disease vascu...,MAPK mitogen-activated protein kinase 14 Seri...,39,0,0.0,0,True
212157,EFO_0006792,ENSG00000112062,0.1,1,Lewy body dementia cortical Lewy body disease ...,MAPK mitogen-activated protein kinase 14 Seri...,7,0,0.0,0,True
116802,EFO_0002893,ENSG00000106991,0.1,1,"choriocarcinoma choriocarcinoma, malignant cho...",ENG endoglin Vascular endothelium glycoprotei...,3,0,0.0,0,True
163556,EFO_0004238,ENSG00000109339,0.3,1,"hearing loss hearing loss disorder loss, heari...",MAPK mitogen-activated protein kinase 10 Seri...,16,1,0.0,0,True
497133,MONDO_0010797,ENSG00000181019,0.1,1,Pearson syndrome Pearson marrow-pancreas syndr...,NQO NAD(P)H quinone dehydrogenase 1 Flavin-co...,1,0,0.0,0,True
541550,MONDO_0015517,ENSG00000115020,0.0,1,common variable immunodeficiency sporadic hypo...,"PIKFYVE phosphoinositide kinase, FYVE-type zin...",2,0,0.0,0,True
480623,MONDO_0009723,ENSG00000181019,0.1,1,Leigh syndrome infantile necrotizing encephalo...,NQO NAD(P)H quinone dehydrogenase 1 Flavin-co...,1,0,0.0,0,True
654739,Orphanet_43,ENSG00000101986,0.6,1,X-linked adrenoleukodystrophy X-linked ALD X-A...,ABCD ATP binding cassette subfamily D member 1...,3,0,0.0,0,True
577320,MONDO_0019046,ENSG00000101986,0.3,1,leukodystrophy HLD hypomyelinating leukodystro...,ABCD ATP binding cassette subfamily D member 1...,2,0,0.0,0,True


In [193]:
df.query('mistake & label==0').sort_values(["score","pred"],ascending=False).round(1).head(10)

,diseaseId,targetId,score,label,disease_text,target_text,num_known_targets,in_aba,pred,pred_label,mistake
313596,EFO_1001010,ENSG00000183454,0.8,0,Landau-Kleffner syndrome acquired epileptic ap...,GRINA glutamate ionotropic receptor NMDA type ...,11,1,0.5,1,True
455274,MONDO_0008224,ENSG00000007314,0.8,0,hyperkalemic periodic paralysis HYPP adynamia ...,SCNA sodium voltage-gated channel alpha subuni...,4,0,0.7,1,True
661543,Orphanet_97,ENSG00000141837,0.8,0,Familial paroxysmal ataxia Episodic ataxia typ...,CACNAA calcium voltage-gated channel subunit a...,40,0,0.9,1,True
538219,MONDO_0015263,ENSG00000183873,0.8,0,Brugada syndrome idiopathic ventricular fibril...,SCNA sodium voltage-gated channel alpha subuni...,40,1,0.7,1,True
478568,MONDO_0009672,ENSG00000172062,0.8,0,"spinal muscular atrophy, type III SMA type 3 K...","SMN survival of motor neuron 1, telomeric The...",42,0,0.6,1,True
630269,MONDO_0100062,ENSG00000145864,0.8,0,developmental and epileptic encephalopathy inf...,GABRB gamma-aminobutyric acid type A receptor ...,7,1,0.5,1,True
455249,MONDO_0008223,ENSG00000007314,0.8,0,hypokalemic periodic paralysis HypoPP HKPP per...,SCNA sodium voltage-gated channel alpha subuni...,5,0,0.6,1,True
415792,MONDO_0004976,ENSG00000127824,0.8,0,amyotrophic lateral sclerosis Lou Gehrig's dis...,TUBAA tubulin alpha 4a Tubulin is the major c...,145,0,1.0,1,True
24143,EFO_0000305,ENSG00000066468,0.7,0,breast carcinoma breast carcinoma mammary carc...,FGFR fibroblast growth factor receptor 2 Tyro...,199,1,0.7,1,True
558417,MONDO_0017276,ENSG00000186868,0.7,0,frontotemporal dementia multiple system tauopa...,MAPT microtubule associated protein tau Promo...,66,1,0.5,1,True


In [194]:
fold_metrics.mean().round(3)

fold         2.000
roc_auc      0.948
prauc        0.772
accuracy     0.939
f1           0.692
precision    0.711
recall       0.676
repeat       2.000
dtype: float64

In [195]:
fold_metrics.std().round(3)

fold         1.443
roc_auc      0.006
prauc        0.023
accuracy     0.005
f1           0.021
precision    0.020
recall       0.034
repeat       1.443
dtype: float64

In [196]:
oof_df = oof_pred
# Evaluate only on rows that actually received OOF preds
mask = oof_df["pred"].notna()
y_oof = oof_df.loc[mask, "label"].values
p_oof = oof_df.loc[mask, "pred"].values
pred_lbl = (p_oof > 0.5).astype(int)

from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
overall = {
    "accuracy": accuracy_score(y_oof, pred_lbl),
    "roc_auc": roc_auc_score(y_oof, p_oof),
    "prauc": average_precision_score(y_oof, p_oof),
    "f1": f1_score(y_oof, pred_lbl),
    "precision": precision_score(y_oof, pred_lbl),
    "recall": recall_score(y_oof, pred_lbl),
}
overall

{'accuracy': 0.9403061124502714,
 'roc_auc': 0.9500027848950302,
 'prauc': 0.7781460629797888,
 'f1': 0.6979327179800138,
 'precision': 0.7197520375090468,
 'recall': 0.6773973819818753}

In [197]:
df_assoc_direct.columns

Index(['diseaseId', 'targetId', 'score_affected_pathway', 'score_animal_model',
       'score_direct', 'score_genetic_association', 'score_known_drug',
       'score_literature', 'score_rna_expression', 'score_somatic_mutation'],
      dtype='object')

In [198]:
df.loc[df[["pred","label"]].max(axis=1)>0.5]

,diseaseId,targetId,score,label,disease_text,target_text,num_known_targets,in_aba,pred,pred_label,mistake
212,DOID_10113,ENSG00000113327,0.036958,1,trypanosomiasis Trypanosoma caused disease or ...,GABRG gamma-aminobutyric acid type A receptor ...,11,1,0.831332,1,False
214,DOID_10113,ENSG00000113578,0.323382,1,trypanosomiasis Trypanosoma caused disease or ...,FGF fibroblast growth factor 1 Plays an impor...,11,1,0.021622,0,True
219,DOID_10113,ENSG00000115758,0.357302,1,trypanosomiasis Trypanosoma caused disease or ...,ODC ornithine decarboxylase 1 Catalyzes the f...,11,0,0.271206,0,True
232,DOID_10113,ENSG00000147955,0.036958,1,trypanosomiasis Trypanosoma caused disease or ...,SIGMAR sigma non-opioid intracellular receptor...,11,0,0.244505,0,True
234,DOID_10113,ENSG00000151834,0.036958,1,trypanosomiasis Trypanosoma caused disease or ...,GABRA gamma-aminobutyric acid type A receptor ...,11,1,0.819765,1,False
...,...,...,...,...,...,...,...,...,...,...,...
659196,Orphanet_848,ENSG00000121966,0.036958,1,Beta-thalassemia Beta-thalassemia (BT) is cha...,CXCR C-X-C motif chemokine receptor 4 Recepto...,25,0,0.144873,0,True
659237,Orphanet_848,ENSG00000160191,0.036958,1,Beta-thalassemia Beta-thalassemia (BT) is cha...,PDEA phosphodiesterase 9A Specifically hydrol...,25,0,0.487243,0,True
660883,Orphanet_912,ENSG00000196664,0.073916,1,Zellweger syndrome Cerebrohepatorenal syndrome...,TLR toll like receptor 7 Endosomal receptor t...,2,0,0.124730,0,True
661294,Orphanet_95157,ENSG00000023330,0.439368,1,Acute hepatic porphyria Acute hepatic porphyr...,ALAS 5'-aminolevulinate synthase 1 Catalyzes ...,1,0,0.048201,0,True


In [199]:
df2 = df.merge(df_assoc_direct,on=["diseaseId","targetId"])

# ## Keep "positve" cases: real or predicted
# df2 = df2.loc[df2[['label', 'pred_label']].max(axis=1)>0].round(4)
df2

,diseaseId,targetId,score,label,disease_text,target_text,num_known_targets,in_aba,pred,pred_label,mistake,score_affected_pathway,score_animal_model,score_direct,score_genetic_association,score_known_drug,score_literature,score_rna_expression,score_somatic_mutation
0,DOID_0050890,ENSG00000004948,0.002957,0,synucleinopathy alpha Synucleinopathies synucl...,CALCR calcitonin receptor G protein-coupled r...,0,1,0.032532,0,False,NaN,NaN,0.002957,NaN,NaN,0.024317,NaN,NaN
1,DOID_0050890,ENSG00000006210,0.004507,0,synucleinopathy alpha Synucleinopathies synucl...,CXCL C-X3-C motif chemokine ligand 1 Chemokin...,0,1,0.004019,0,False,NaN,NaN,0.004507,NaN,NaN,0.037067,NaN,NaN
2,DOID_0050890,ENSG00000007171,0.009979,0,synucleinopathy alpha Synucleinopathies synucl...,NOS nitric oxide synthase 2 Produces nitric o...,0,0,0.004452,0,False,NaN,NaN,0.009979,NaN,NaN,0.082071,NaN,NaN
3,DOID_0050890,ENSG00000007952,0.008685,0,synucleinopathy alpha Synucleinopathies synucl...,NOX NADPH oxidase 1 NADPH oxidase that cataly...,0,0,0.005626,0,False,NaN,NaN,0.008685,NaN,NaN,0.071432,NaN,NaN
4,DOID_0050890,ENSG00000010610,0.005862,0,synucleinopathy alpha Synucleinopathies synucl...,CD CD4 molecule Integral membrane glycoprotei...,0,0,0.016121,0,False,NaN,NaN,0.005862,NaN,NaN,0.048212,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
663346,Orphanet_99946,ENSG00000101986,0.054816,0,Autosomal dominant Charcot-Marie-Tooth disease...,ABCD ATP binding cassette subfamily D member 1...,0,0,0.003915,0,False,NaN,0.450841,0.054816,NaN,NaN,NaN,NaN,NaN
663347,Orphanet_99947,ENSG00000101986,0.033809,0,Autosomal dominant Charcot-Marie-Tooth disease...,ABCD ATP binding cassette subfamily D member 1...,0,0,0.002175,0,False,NaN,0.278068,0.033809,NaN,NaN,NaN,NaN,NaN
663348,Orphanet_99947,ENSG00000129244,0.032190,0,Autosomal dominant Charcot-Marie-Tooth disease...,ATPB ATPase Na+/K+ transporting subunit beta 2...,0,1,0.003962,0,False,NaN,0.264754,0.032190,NaN,NaN,NaN,NaN,NaN
663349,UBERON_0000104,ENSG00000135744,0.002217,0,life cycle life entire lifespan entire life cy...,AGT angiotensinogen Essential component of th...,0,1,0.013084,0,False,NaN,NaN,0.002217,NaN,NaN,0.018238,NaN,NaN


In [200]:
print("corr - in positives interactions (real or predicted)")
df2.select_dtypes(["number","bool"]).corr().round(2).loc[['score', 'label', 'pred', 'pred_label',
       'mistake',  'score_animal_model',
       'score_direct', 'score_genetic_association', 'score_known_drug',]]

corr - in positives interactions (real or predicted)


,score,label,num_known_targets,in_aba,pred,pred_label,mistake,score_affected_pathway,score_animal_model,score_direct,score_genetic_association,score_known_drug,score_literature,score_rna_expression,score_somatic_mutation
score,1.00,0.44,0.18,0.04,0.34,0.31,0.17,0.55,0.29,1.00,0.93,0.93,0.34,0.13,0.83
label,0.44,1.00,0.29,0.02,0.72,0.67,0.38,0.11,0.05,0.44,0.05,0.01,0.17,0.01,0.10
pred,0.34,0.72,0.48,0.04,1.00,0.88,0.32,0.05,0.02,0.34,-0.10,0.08,0.11,0.05,0.03
pred_label,0.31,0.67,0.33,0.03,0.88,1.00,0.30,0.06,0.02,0.31,-0.07,0.07,0.07,-0.00,0.02
mistake,0.17,0.38,0.21,0.05,0.32,0.30,1.00,0.03,0.03,0.17,-0.01,-0.07,0.10,0.01,0.04
score_animal_model,0.29,0.05,0.03,-0.02,0.02,0.02,0.03,0.09,1.00,0.29,0.04,0.11,0.07,-0.21,0.11
score_direct,1.00,0.44,0.18,0.04,0.34,0.31,0.17,0.55,0.29,1.00,0.93,0.93,0.34,0.13,0.83
score_genetic_association,0.93,0.05,-0.10,0.03,-0.10,-0.07,-0.01,0.20,0.04,0.93,1.00,0.14,0.07,0.01,0.32
score_known_drug,0.93,0.01,0.02,0.06,0.08,0.07,-0.07,0.03,0.11,0.93,0.14,1.00,0.06,-0.03,0.11


In [201]:
df2.select_dtypes(["number","bool"]).groupby("label").mean().round(2)

,score,num_known_targets,in_aba,pred,pred_label,mistake,score_affected_pathway,score_animal_model,score_direct,score_genetic_association,score_known_drug,score_literature,score_rna_expression,score_somatic_mutation
label,,,,,,,,,,,,,,
0,0.03,43.02,0.24,0.09,0.03,0.03,0.62,0.36,0.03,0.42,0.15,0.10,0.08,0.39
1,0.18,130.94,0.27,0.65,0.68,0.32,0.68,0.42,0.18,0.47,0.27,0.21,0.09,0.43


### Add all druggable genome X diseases as additional candidates data
* Maybe filter for subset of diseases due to size!
* Need to filter for cases not already done
* We'll get predictions from prev.trained model on these.
* Keep all druggable genome targets (not already covered) togethjer with diseases.

* will need to add score from: associations_df 

In [202]:
# df_learn = df_learn.merge(disease_df[["diseaseId","disease_text_embed"]],on="diseaseId")#,left_on="diseaseId",right_index=True)
# df_learn = df_learn.merge(target_df[["targetId","target_text_embed"]],on="targetId")#,left_on="targetId",right_index=True)
# df_learn.rename(columns={"disease_text_embed":"disease_text","target_text_embed":"target_text"},inplace=True,errors="ignore")


# disease_df
# target_df

In [203]:
target_df_druggable = target_df.query("targetId in @druggable_genome_list").reset_index(drop=True)
print(target_df_druggable.shape[0])
## drop cases already covered - assuming any in relationship (and not "target+disease"):
## should drop around 1300~
target_df_druggable = target_df_druggable.query("targetId not in @df_all.targetId").reset_index(drop=True)
print(target_df_druggable.shape[0]) # 3080

4316
3080


####  Warning! We are taking only cases with any sort of evidence/score for the pair - we'll be missing cases with 0 evidence at all. 
* That's still most!
* We could use indirect evidence OR simply get "all to all" merge/join!!
* 
* This is IMPORTANT ! 

In [204]:
# #### NOTE! These are only cases with some sort of direct evidence! Others will be missing: fill as 0.  (+- look at indirect)
# associations_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'association_overall_direct')).query("targetId in @druggable_genome_list")
# associations_df # 1.6M direct candidates

### done naively, this is a huge number of candidates! 120 M !
* We will take as candidates those with any indirect association , to speed thigns up : ~ 4M candidates! 

In [205]:
indir_associations_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'association_by_overall_indirect'),columns=['diseaseId', 'targetId', 'score']).query("targetId in @druggable_genome_list")
print(indir_associations_df.shape[0])
indir_associations_df = indir_associations_df[~indir_associations_df['diseaseId'].isin(ids_to_remove)] # drop some unwanted diseases, like cell proliferation, measurement
print(indir_associations_df.shape[0])
indir_associations_df

if FAST_RUN:
    indir_associations_df = indir_associations_df.head(n=80_000)
# indirect associations: more liberal. Can use as starting point.
## Still need score from direct associations score df (and fillna as 0)

3994266
3398870


In [206]:
indir_associations_df.drop_duplicates(subset=['diseaseId', 'targetId']).shape

(3398870, 3)

In [207]:
# --- 2. The anti-join by 2 keys One-Liner ---
# We use a left merge with an indicator.
# Rows from 'df' that find a match in 'indir_associations_df' will be marked as 'both'.
# Rows from 'df' that DO NOT find a match will be marked as 'left_only'.
# We then query for 'left_only' (the rows we want to keep) and drop the helper column.
# We also drop_duplicates from the exclusion frame for efficiency.

df_cands_novel = indir_associations_df.merge(
    df_all[['diseaseId', 'targetId']].drop_duplicates(),
    on=['diseaseId', 'targetId'],
    how='left',
    indicator=True
).query('_merge == "left_only"').drop(columns='_merge')
df_cands_novel

,diseaseId,targetId,score
0,DOID_0050890,ENSG00000000971,0.003696
1,DOID_0050890,ENSG00000001084,0.031799
2,DOID_0050890,ENSG00000002726,0.001478
3,DOID_0050890,ENSG00000004468,0.029443
4,DOID_0050890,ENSG00000004478,0.002217
...,...,...,...
3398859,Orphanet_99947,ENSG00000184381,0.044518
3398860,Orphanet_99947,ENSG00000184451,0.035812
3398862,Orphanet_99947,ENSG00000188157,0.051540
3398866,Orphanet_99947,ENSG00000197746,0.051103


In [208]:
assert df_cands_novel.drop_duplicates(subset=['diseaseId', 'targetId']).shape[0] == df_cands_novel.shape[0]

In [209]:
# %%time
# df_cands_novel = target_df_druggable[["targetId","target_text_embed"]].merge(disease_df[["diseaseId","disease_text_embed"]],how="cross") # 120M ! huge! 

In [210]:
# disease_df[["diseaseId","disease_text_embed","name"]].nunique() # 38K

# df_learn.merge(disease_df[["diseaseId","disease_text_embed"]],on="diseaseId")#,left_on="diseaseId",right_index=True)
# df_learn = df_learn.merge(target_df[["targetId","target_text_embed"]],on="targetId")

###### TODO: Why are there different # cnadiadtes?

In [211]:
s1 = df_cands_novel.shape[0]
print(s1,"# cands pre merge")
df_cands_novel = df_cands_novel.merge(disease_df[["diseaseId","disease_text_embed"]],on="diseaseId")
df_cands_novel = df_cands_novel.merge(target_df[["targetId","target_text_embed"]],on="targetId")
df_cands_novel.rename(columns={"disease_text_embed":"disease_text","target_text_embed":"target_text"},inplace=True,errors="ignore")
df_cands_novel

if s1 != df_cands_novel.shape[0]:
    print("Warning: # candidates changed")
    print(s1 - df_cands_novel.shape[0])

2803207 # cands pre merge
30632


In [212]:
import numpy as np
import pandas as pd
import gc

def predict_in_chunks(
    df: pd.DataFrame,
    model,
    feature_fn,
    chunk_size: int = 250_000,
    feature_kwargs: dict | None = None
) -> pd.DataFrame:
    """
    Lazily feature-ify and score df in chunks, then return df with a new 'pred' column.
    Assumes model.predict(..., prediction_type='Probability') returns either shape (n,2) or (n,).
    """
    feature_kwargs = feature_kwargs or {}

    n = len(df)
    preds = np.full(n, np.nan, dtype=float)

    # map original row index -> absolute position for stable write-back
    pos = pd.Series(np.arange(n), index=df.index, copy=False)

    start = 0
    while start < n:
        end = min(start + chunk_size, n)

        # slice without copying columns unnecessarily
        chunk = df.iloc[start:end]

        # build features for this chunk
        X = feature_fn(chunk, **feature_kwargs)
        display(X.iloc[:,15:19])
        # display(X.iloc[:,10:19].dtypes)
        # score; handle (n,2) vs (n,) outputs
        out = model.predict(X, prediction_type="Probability")
        proba = out[:, 1] if getattr(out, "ndim", 1) == 2 else np.asarray(out, dtype=float)

        # align-by-index in case feature_fn reorders rows
        idx_positions = pos.loc[X.index].to_numpy()
        preds[idx_positions] = proba

        # free memory
        del X, out, proba, idx_positions, chunk
        gc.collect()

        start = end

    # attach predictions to a shallow copy of the original frame
    result = df.copy()
    result["pred"] = preds
    return result

In [213]:
%%time
# choose the same flags you already use for training/inference
feature_kwargs = dict(
    # get_embeddings=False,
    # add_target_feats=True,
    merge_aba=False,
    # merge_mouse_pheno=False,
    # merge_depmap_essentiality=True,
    # get_network_embeds=True,
    # embed_cols=None,
)


df_scored = predict_in_chunks(
    df=df_cands_novel,
    model=model,
    # model = CatBoostClassifier(**cb_params),
    feature_fn=get_cb_x,
    chunk_size=400_000,
    feature_kwargs=feature_kwargs
)

(400000, 579)


,score_syn,score_mis,score_lof,subcellularLocations
_rid,,,,
0,-1.763700,0.997350,8.606100e-01,"['Secreted', 'Vesicles', 'Predicted to be secr..."
1,0.587060,2.549400,4.271800e-01,"['Nucleoplasm', 'Cytosol', 'Nucleoli']"
2,0.675460,0.758260,6.723600e-08,"['Secreted', 'Cell membrane']"
3,0.961390,-0.064677,2.312900e-06,"['Cell surface', 'Membrane', 'Plasma membrane']"
4,-1.060800,1.208200,1.553900e-01,"['Cytoplasm', 'Mitochondrion', 'Nucleus', 'Cel..."
...,...,...,...,...
399995,0.092598,3.404300,9.992800e-01,"['Cell membrane', 'Cell projection', 'Vesicles..."
399996,-0.251610,0.549820,7.199000e-03,"['Cell membrane', 'Membrane raft', 'Secreted',..."
399997,1.176500,-0.177050,8.683100e-04,"['Cytoplasm', 'Cell junction', 'Focal adhesion..."


CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=18]="['Secreted', 'Vesicles', 'Predicted to be secreted']": Cannot convert '['Secreted', 'Vesicles', 'Predicted to be secreted']' to float

##### Add (indirect?) association scores, by source, as metadata
* could use direct or indirect
* total score and/or by source (note by known drug)

* Or use the (already loaded) direct sources, assuming it has coverage (and wasn't filtered)?

In [214]:
# df_assoc_indir = reset_pivoted_multiindex(pd.read_parquet(os.path.join(Config.DATA_DIR, 'association_by_datatype_ןמdirect'),columns=['diseaseId', 'targetId', 'score']).pivot(index=["diseaseId","targetId"], columns='datatypeId',values=["score"]))
# df_assoc_indir

In [215]:
# 1. Group df and sum 'label'
num_known_targets_counts = df.groupby("diseaseId")["label"].sum()
# 2. Map the sums to df_scored using 'diseaseId' and fill NaNs with 0
df_scored["num_known_targets"] = df_scored["diseaseId"].map(num_known_targets_counts).fillna(0)
# df_scored["num_known_targets"] = df.groupby("diseaseId")["label"].transform("sum") # for sorting by cases without known
df_scored["in_aba"] = df_scored["targetId"].isin(df_aba.index).astype(int)
df_scored = df_scored.merge(df_assoc_direct,on=["diseaseId","targetId"],how="left").dropna(axis=1,thresh=100)
df_scored["pred_label"] = (df_scored["pred"]>0.5).astype(int)

NameError: name 'df_scored' is not defined

In [ ]:
df_scored

In [ ]:
df2.select_dtypes("number").mean().round(3)

In [ ]:
df_scored.select_dtypes("number").mean().round(3)

##### Q: We see "novel" candidates with possible associations - reasonable?
* these are from indirect assoc; but score is from direct
* maybe drug not same as clinicial trial! (our real criteria for default label)

In [ ]:
print(df2.shape[0])
# _1 = df2.query("(label>0)|(pred_label>0)").dropna(axis=1,thresh=50).round(3).sort_values(["label","pred","score","diseaseId"],ascending=[True,False,True,True]).drop(columns=["mistake"],errors="ignore")
# print(_1.shape)
print(df_scored.shape[0])
# _2 = df_scored.query("pred_label>0").dropna(axis=1,thresh=50).round(3).sort_values(["pred","score","diseaseId"],ascending=[False,True,True])
# print(_2.shape)

# _1.to_csv("positive_preds_trainData.csv")
# _2.to_csv("positive_preds_NovelIndirData.csv")

df_novel_pos_predictions = pd.concat([df2.query("(label>0)|(pred_label>0)").dropna(axis=1,thresh=50).round(3).sort_values(["label","pred","score","diseaseId"],ascending=[True,False,True,True]).drop(columns=["mistake"],errors="ignore"),
                                 df_scored.query("pred_label>0").dropna(axis=1,thresh=50).round(3).sort_values(["pred","score","diseaseId"],ascending=[False,True,True])],ignore_index=True)
df_novel_pos_predictions.drop(columns=['disease_text', 'target_text'],inplace=True,errors="ignore") # drop long terxt descriptions for final output - save on space
df_novel_pos_predictions.drop_duplicates(subset=["diseaseId","targetId"],inplace=True)
print(df_novel_pos_predictions.shape)
df_novel_pos_predictions = df_novel_pos_predictions.merge(disease_df[['diseaseId',"name"]].rename(columns={"name":"disease_name"}),on="diseaseId",how="left")
df_novel_pos_predictions = df_novel_pos_predictions.merge(target_df[['targetId', 'approvedSymbol', 'biotype', 'approvedName']].rename(columns={"approvedName":"Gene_name"}),on="targetId",how="left")
print(df_novel_pos_predictions.shape[0])
## reorder columns
df_novel_pos_predictions = df_novel_pos_predictions[[ 'disease_name', 'approvedSymbol', 
   'Gene_name', 'label',  'pred', 'pred_label', 'diseaseId', 'targetId', 'num_known_targets',
   'in_aba','score_direct', 'score_known_drug', #  'score',
   'score_literature', 'score_affected_pathway',
   'score_genetic_association', 'score_rna_expression',
   'score_somatic_mutation','num_known_targets',
   'biotype','in_aba',]].sort_values(['label',"pred","score_direct","diseaseId"],ascending=[True,False,False,True])
display(df_novel_pos_predictions)

In [ ]:
# if SAVE_NOVEL_PREDICTION_CANDIDATES:
#     df_novel_pos_predictions.to_csv("positive_preds_train_and_novelIndirect.csv",index=False)

In [ ]:
df_novel_pos_predictions.dropna(subset=["score_direct"],axis=0)